# Multi-Stage Heuristic and NLI Ensemble for Claim-Level Hallucination Control in Retrieval-Augmented LLM Medical QA
## A Reproducible Pilot Study — NICE NG28

Refactored, de-duplicated workflow in **16 independent stages** (the original 12-stage
core pipeline, plus 4 stages recovered from the archive: reference-claim verification,
adversarial evaluation, NLI-based verification, and threshold sensitivity analysis).

| Stage | Purpose | Main output |
|---|---|---|
| 1 | Configuration and constants | config variables |
| 2 | Corpus loading | `corpus_records` |
| 3 | Corpus validation | `corpus_validation_report` |
| 4 | Dense embedding index | `chunk_embeddings` |
| 5 | Query preprocessing utilities | `tokenize`, `content_stems`, ... |
| 6 | Lexical and metadata reranking | `lexical_bonus` |
| 7 | Top-k retrieval function | `retrieve_top_k` |
| 8 | Retrieval evaluation (smoke test + main) | `retrieval_results`, `hit_at_3_score` |
| 9 | Evidence display and export | `manual_claim_records` (gold) |
| 10 | Extractive QA answer candidates | `qa_candidate_answers` |
| 11 | Rule-based claim decomposition | `atomic_claims` |
| 12 | Claim verification and final report | `claim_verification_records`, `final_report.md` |
| 13 | Stage 4 V2 — Reference-claim verification (human gold vs automated) | `REFERENCE_CLAIM_ROWS`, per-status P/R/F1 |
| 14 | Adversarial claims evaluation (7 deliberately contradictory claims) | `ADVERSARIAL_CLAIMS`, `ADVERSARIAL_RESULTS` |
| 15 | Stage 4 V3 — NLI-based claim verification (`cross-encoder/nli-MiniLM2-L6-H768`) | `STAGE4_V3_RECORDS` |
| 16 | Sensitivity analysis (thresholds 0.70 / 0.50 / 0.45) + heuristic-OR-NLI ensemble | `SENSITIVITY_ROWS` |

**Scope statement (must not be softened).** Stage 12 is a **heuristic** verifier. It uses
only (a) the retrieval similarity score, (b) content-stem lexical overlap, and
(c) numerical consistency. It does **not** perform natural-language inference, it does not
model entailment or contradiction, and it is not a clinical safety check. Corpus `text`,
`chunk_id`, `document_id`, `section`, `recommendation_number`, `page`, `source_url`, the
evaluation questions, and the manually curated gold labels are reproduced unchanged from
the source notebook. **Stage 13** and **Stage 14** reuse this exact same Stage 12
heuristic verifier (no new decision logic) — Stage 13 runs it against human-written claim
text instead of machine-extracted claims, and Stage 14 runs it against 7 deliberately
contradictory claims. **Stage 15 is the only stage that performs true natural-language
inference** (entailment / contradiction / neutral), via a local NLI model. It is kept
separate from Stage 12 so Stage 12's "heuristic only" scope statement above stays
accurate. **Stage 16 performs no new model inference**; it only re-applies Stage 15's
already-computed scores at alternative thresholds.

**Manual vs automated.** `manual_claim_records` (Stage 9) are human-written gold/reference
records. `claim_verification_records` (Stage 12) are automated pipeline output. They are
kept in separate variables and separate files and are never merged. Stage 13's
`REFERENCE_CLAIM_ROWS` and Stage 15's `STAGE4_V3_RECORDS` compare against
`manual_claim_records` as the gold reference but likewise never overwrite or merge into it.

No paid API, API key, or hosted inference service is used. Everything — including the
Stage 15 NLI model — runs locally.


---
# Stage 1 — Configuration and constants

**Input:** none. **Output:** all tunable constants used downstream.
Nothing here performs computation, so this stage is safe to re-run at any time.

In [ ]:
!pip install -q sentence-transformers "transformers[torch]" pandas

In [ ]:
"""Stage 1: every tunable constant in one place."""

# --- Models (both are free, local, and downloaded from the Hugging Face hub) ---
EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
QA_MODEL_NAME = "distilbert-base-cased-distilled-squad"

# --- Retrieval ---
TOP_K = 3
EXPECTED_CORPUS_SIZE = 18
EXPECTED_EMBEDDING_DIM = 384

# --- Lexical rerank weights (unchanged from the source notebook) ---
STEM_OVERLAP_WEIGHT = 0.22
DISTINCTIVE_STEM_WEIGHT = 0.10
DISTINCTIVE_STEM_CAP = 0.20
NUMBER_OVERLAP_WEIGHT = 0.18
META_OVERLAP_WEIGHT = 0.08
DISTINCTIVE_STEM_MIN_LEN = 6

# --- Extractive QA ---
QA_MAX_ANSWER_LEN = 40
QA_TOP_K_LOGITS = 20
QA_MAX_SEQ_LEN = 384

# --- Stage 12 heuristic verifier thresholds (documented and adjustable) ---
SIMILARITY_THRESHOLD = 0.35    # below this -> Insufficient Evidence
TERM_OVERLAP_THRESHOLD = 0.50  # fraction of claim content stems found in evidence
CLAIM_MIN_CONTENT_STEMS = 4    # completeness threshold for a claim

ALLOWED_STATUSES = {"Supported", "Unsupported", "Insufficient Evidence"}

# --- Output file names ---
MANUAL_CLAIMS_CSV = "manual_claim_records.csv"
MANUAL_CLAIMS_JSON = "manual_claim_records.json"
AUTOMATED_CLAIMS_CSV = "automated_claim_verification.csv"
AUTOMATED_CLAIMS_JSON = "automated_claim_verification.json"
FINAL_REPORT_MD = "final_report.md"
FINAL_SUMMARY_JSON = "final_summary.json"

print("Stage 1 configuration loaded.")
print(f"  embedding model: {EMBEDDING_MODEL_NAME}")
print(f"  QA model:        {QA_MODEL_NAME}")
print(f"  top_k:           {TOP_K}")

# =====================================================================
# Appended for Stages 13-16 (Stage 4 V2 / adversarial / Stage 4 V3 NLI /
# sensitivity analysis). Nothing above this line was changed -- every
# constant and print statement used by Stages 1-12 is exactly as before.
# =====================================================================

# --- Stage 13 / Stage 14 (Stage 4 V2 + adversarial) output file names ---
STAGE4V2_REFERENCE_CSV = "stage4v2_reference_claim_comparison.csv"
STAGE4V2_REFERENCE_JSON = "stage4v2_reference_claim_comparison.json"
STAGE4V2_ADVERSARIAL_CSV = "stage4v2_adversarial_claims.csv"
STAGE4V2_ADVERSARIAL_JSON = "stage4v2_adversarial_claims.json"

# --- Stage 15 (Stage 4 V3, local NLI model) ---
STAGE4_V3_MODEL_NAME = "cross-encoder/nli-MiniLM2-L6-H768"
NLI_ENTAILMENT_THRESHOLD = 0.70     # >= this AND strongest relation -> Supported
NLI_CONTRADICTION_THRESHOLD = 0.70  # >= this AND strongest relation -> Unsupported
STAGE4V3_NLI_CSV = "stage4v3_nli_comparison.csv"
STAGE4V3_NLI_JSON = "stage4v3_nli_comparison.json"
STAGE4V3_NLI_SUMMARY = "stage4v3_nli_summary.json"

# --- Stage 16 (sensitivity analysis + heuristic-OR-NLI ensemble) ---
# 0.70 = primary reference point (must match NLI_ENTAILMENT_THRESHOLD /
# NLI_CONTRADICTION_THRESHOLD above); 0.50 and 0.45 are exploratory, post-hoc.
SENSITIVITY_THRESHOLDS = [0.70, 0.50, 0.45]
SENSITIVITY_CSV = "sensitivity_threshold_analysis.csv"
SENSITIVITY_SUMMARY = "sensitivity_summary.json"
SENSITIVITY_REPORT_MD = "sensitivity_report.md"

print("Stage 13-16 configuration loaded.")
print(f"  NLI model:                   {STAGE4_V3_MODEL_NAME}")
print(f"  NLI entailment threshold:    {NLI_ENTAILMENT_THRESHOLD}")
print(f"  NLI contradiction threshold: {NLI_CONTRADICTION_THRESHOLD}")
print(f"  sensitivity thresholds:      {SENSITIVITY_THRESHOLDS}")


---
# Stage 2 — Corpus loading

**Input:** the embedded NICE NG28 JSON string. **Output:** `corpus_records` (18 dicts).

The guideline `text` is used exactly as published and is never rewritten, normalised,
or truncated anywhere in this notebook. `chunk_id`, `document_id`, `section`,
`recommendation_number`, `page`, and `source_url` are reproduced unchanged.

In [ ]:
import json
import re
from typing import Any

import numpy as np

CORPUS_JSON = r'''
[
  {
    "chunk_id": "NICE_NG28_001",
    "document_id": "NICE_NG28",
    "section": "Dietary advice",
    "recommendation_number": "1.3.1",
    "page": 10,
    "source_url": "https://www.nice.org.uk/guidance/ng28",
    "text": "Provide individualised and ongoing nutritional advice from a healthcare professional with specific expertise and competencies in nutrition."
  },
  {
    "chunk_id": "NICE_NG28_002",
    "document_id": "NICE_NG28",
    "section": "Dietary advice",
    "recommendation_number": "1.3.3",
    "page": 10,
    "source_url": "https://www.nice.org.uk/guidance/ng28",
    "text": "Encourage adults with type 2 diabetes to follow the same healthy eating advice as the general population, which includes:\n• eating high-fibre, low-glycaemic-index sources of carbohydrate, such as fruit, vegetables, wholegrains and pulses\n• choosing low-fat dairy products\n• eating oily fish\n• controlling their intake of saturated and trans fatty acids."
  },
  {
    "chunk_id": "NICE_NG28_003",
    "document_id": "NICE_NG28",
    "section": "Dietary advice",
    "recommendation_number": "1.3.6",
    "page": 10,
    "source_url": "https://www.nice.org.uk/guidance/ng28",
    "text": "Individualise recommendations for carbohydrate and alcohol intake, and meal patterns. Make reducing the risk of hypoglycaemia a particular aim for people using insulin or an insulin secretagogue."
  },
  {
    "chunk_id": "NICE_NG28_004",
    "document_id": "NICE_NG28",
    "section": "Dietary advice",
    "recommendation_number": "1.3.8",
    "page": 11,
    "source_url": "https://www.nice.org.uk/guidance/ng28",
    "text": "Discourage adults with type 2 diabetes from using foods marketed specifically for people with diabetes."
  },
  {
    "chunk_id": "NICE_NG28_005",
    "document_id": "NICE_NG28",
    "section": "Blood glucose management and HbA1c",
    "recommendation_number": "1.5.1",
    "page": 12,
    "source_url": "https://www.nice.org.uk/guidance/ng28",
    "text": "Measure HbA1c levels in adults with type 2 diabetes every:\n• 3 to 6 months (tailored to individual needs) until HbA1c is stable on unchanging therapy\n• 6 months once the HbA1c level and blood glucose lowering therapy are stable."
  },
  {
    "chunk_id": "NICE_NG28_006",
    "document_id": "NICE_NG28",
    "section": "Blood glucose management and HbA1c",
    "recommendation_number": "1.5.7",
    "page": 13,
    "source_url": "https://www.nice.org.uk/guidance/ng28",
    "text": "For adults whose type 2 diabetes is managed either by healthy living and diet, or healthy living and diet combined with an initial medication regimen that is not associated with hypoglycaemia (see the section on initial medicines), support them to aim for an HbA1c level of 48 mmol/mol (6.5%). For adults on a medicine associated with hypoglycaemia, support them to aim for an HbA1c level of 53 mmol/mol (7.0%). [2015, amended 2026]"
  },
  {
    "chunk_id": "NICE_NG28_007",
    "document_id": "NICE_NG28",
    "section": "Blood glucose management and HbA1c",
    "recommendation_number": "1.5.8",
    "page": 13,
    "source_url": "https://www.nice.org.uk/guidance/ng28",
    "text": "In adults with type 2 diabetes, if HbA1c levels are not adequately controlled by the initial medication regimen and rise to 58 mmol/mol (7.5%) or higher:\n• reinforce advice about diet, healthy living and adherence to medicines and\n• support the person to aim for an HbA1c level of 53 mmol/mol (7.0%) and\n• intensify medicines. [2015, amended 2026]"
  },
  {
    "chunk_id": "NICE_NG28_008",
    "document_id": "NICE_NG28",
    "section": "Initial medicines",
    "recommendation_number": "1.13.1",
    "page": 32,
    "source_url": "https://www.nice.org.uk/guidance/ng28",
    "text": "For adults with type 2 diabetes and no relevant comorbidity, offer:\n• modified-release metformin, and\n• an SGLT-2 inhibitor."
  },
  {
    "chunk_id": "NICE_NG28_009",
    "document_id": "NICE_NG28",
    "section": "Initial medicines",
    "recommendation_number": "1.13.2",
    "page": 32,
    "source_url": "https://www.nice.org.uk/guidance/ng28",
    "text": "If metformin is contraindicated or not tolerated, offer monotherapy with an SGLT-2 inhibitor."
  },
  {
    "chunk_id": "NICE_NG28_010",
    "document_id": "NICE_NG28",
    "section": "Cardiovascular risk",
    "recommendation_number": "1.11.1",
    "page": 28,
    "source_url": "https://www.nice.org.uk/guidance/ng28",
    "text": "Assess the person's current cardiovascular and renal status, and risk of developing cardiovascular disease in the future. [2022, amended 2026]"
  },
  {
    "chunk_id": "NICE_NG28_011",
    "document_id": "NICE_NG28",
    "section": "Cardiovascular risk",
    "recommendation_number": "1.15.1",
    "page": 45,
    "source_url": "https://www.nice.org.uk/guidance/ng28",
    "text": "For adults with type 2 diabetes and atherosclerotic cardiovascular disease, offer:\n• modified-release metformin, and\n• an SGLT-2 inhibitor, and\n• subcutaneous semaglutide (Ozempic), up to 1 mg once a week, for its cardiovascular, renal and glycaemic benefits."
  },
  {
    "chunk_id": "NICE_NG28_012",
    "document_id": "NICE_NG28",
    "section": "Cardiovascular risk",
    "recommendation_number": "1.45.1",
    "page": 119,
    "source_url": "https://www.nice.org.uk/guidance/ng28",
    "text": "Do not offer antiplatelet therapy (aspirin or clopidogrel) to adults with type 2 diabetes without cardiovascular disease."
  },
  {
    "chunk_id": "NICE_NG28_013",
    "document_id": "NICE_NG28",
    "section": "Complications",
    "recommendation_number": "1.37.1",
    "page": 113,
    "source_url": "https://www.nice.org.uk/guidance/ng28",
    "text": "Advise adults with type 2 diabetes at their annual review that:\n• they are at higher risk of periodontitis\n• if they get periodontitis, managing it can improve their blood glucose control and can reduce their risk of hyperglycaemia."
  },
  {
    "chunk_id": "NICE_NG28_014",
    "document_id": "NICE_NG28",
    "section": "Complications",
    "recommendation_number": "1.38.1",
    "page": 116,
    "source_url": "https://www.nice.org.uk/guidance/ng28",
    "text": "Think about a diagnosis of gastroparesis in adults with type 2 diabetes who have erratic blood glucose control or unexplained gastric bloating or vomiting, taking into account possible alternative diagnoses. [2009, amended 2015]"
  },
  {
    "chunk_id": "NICE_NG28_015",
    "document_id": "NICE_NG28",
    "section": "Complications",
    "recommendation_number": "1.41.1",
    "page": 117,
    "source_url": "https://www.nice.org.uk/guidance/ng28",
    "text": "For guidance on preventing and managing foot problems in adults with type 2 diabetes, see NICE's guideline on diabetic foot problems."
  },
  {
    "chunk_id": "NICE_NG28_016",
    "document_id": "NICE_NG28",
    "section": "Complications",
    "recommendation_number": "1.43.1",
    "page": 118,
    "source_url": "https://www.nice.org.uk/guidance/ng28",
    "text": "When adults are diagnosed with type 2 diabetes, refer them immediately to the local eye screening service. [2009, amended 2020]"
  },
  {
    "chunk_id": "NICE_NG28_017",
    "document_id": "NICE_NG28",
    "section": "Monitoring",
    "recommendation_number": "1.6.2",
    "page": 14,
    "source_url": "https://www.nice.org.uk/guidance/ng28",
    "text": "Do not routinely offer self-monitoring of capillary blood glucose levels for adults with type 2 diabetes unless:\n• the person is on insulin or\n• there is evidence of hypoglycaemic episodes or\n• the person is on oral medication that may increase their risk of hypoglycaemia while driving or operating machinery or\n• they are pregnant or are planning to become pregnant (see NICE's guideline on diabetes in pregnancy)."
  },
  {
    "chunk_id": "NICE_NG28_018",
    "document_id": "NICE_NG28",
    "section": "Monitoring",
    "recommendation_number": "1.7.1",
    "page": 15,
    "source_url": "https://www.nice.org.uk/guidance/ng28",
    "text": "Offer intermittently scanned continuous glucose monitoring (isCGM, commonly referred to as 'flash') to adults with type 2 diabetes on multiple daily insulin injections if any of the following apply:\n• they have recurrent hypoglycaemia or severe hypoglycaemia\n• they have impaired hypoglycaemia awareness\n• they have a condition or disability (including a learning disability or cognitive impairment) that means they cannot self-monitor their blood glucose by capillary blood glucose monitoring but could use an isCGM device (or have it scanned for them)\n• they would otherwise be advised to self-measure at least 8 times a day."
  }
]
'''

corpus_records: list[dict[str, Any]] = json.loads(CORPUS_JSON)

print(f"Loaded {len(corpus_records)} records")
print("Fields:", sorted(corpus_records[0].keys()))

---
# Stage 3 — Corpus validation

**Input:** `corpus_records`. **Output:** `corpus_validation_report`, `corpus_is_valid`.

This stage is read-only: it never edits the corpus. It fails loudly if the corpus has
drifted from the verified 18-chunk version the rest of the notebook assumes.

In [ ]:
import hashlib

REQUIRED_CORPUS_FIELDS = [
    "chunk_id", "document_id", "section", "recommendation_number",
    "page", "source_url", "text",
]


def validate_corpus(records: list[dict]) -> dict:
    """Read-only structural check of the corpus. Returns a report dict."""
    problems = []

    if len(records) != EXPECTED_CORPUS_SIZE:
        problems.append(f"Expected {EXPECTED_CORPUS_SIZE} records, found {len(records)}.")

    seen_ids = set()
    for i, row in enumerate(records):
        missing = [f for f in REQUIRED_CORPUS_FIELDS if f not in row]
        if missing:
            problems.append(f"Record {i}: missing field(s) {missing}.")
            continue
        if not str(row["text"]).strip():
            problems.append(f"Record {i} ({row['chunk_id']}): empty text.")
        if row["chunk_id"] in seen_ids:
            problems.append(f"Duplicate chunk_id: {row['chunk_id']}.")
        seen_ids.add(row["chunk_id"])
        if not isinstance(row["page"], int):
            problems.append(f"{row['chunk_id']}: page is not an integer.")
        if not str(row["source_url"]).startswith("http"):
            problems.append(f"{row['chunk_id']}: source_url does not look like a URL.")

    corpus_digest = hashlib.sha256(
        "\n".join(r["chunk_id"] + "|" + r["text"] for r in records).encode("utf-8")
    ).hexdigest()

    return {
        "n_records": len(records),
        "n_unique_chunk_ids": len(seen_ids),
        "sections": sorted({r["section"] for r in records if "section" in r}),
        "corpus_text_sha256": corpus_digest,
        "problems": problems,
        "status": "PASS" if not problems else "FAIL",
    }


corpus_validation_report = validate_corpus(corpus_records)
corpus_is_valid = corpus_validation_report["status"] == "PASS"

print("STAGE 3 - CORPUS VALIDATION")
print("-" * 60)
print(f"records:            {corpus_validation_report['n_records']}")
print(f"unique chunk_ids:   {corpus_validation_report['n_unique_chunk_ids']}")
print(f"sections:           {len(corpus_validation_report['sections'])}")
for s in corpus_validation_report["sections"]:
    print(f"  - {s}")
print(f"corpus text sha256: {corpus_validation_report['corpus_text_sha256'][:16]}...")
print(f"status:             {corpus_validation_report['status']}")
for p in corpus_validation_report["problems"]:
    print("  PROBLEM:", p)

assert corpus_is_valid, "Corpus validation failed - do not continue."

# Lookup used by later stages (chunk_id -> full record).
chunk_lookup = {row["chunk_id"]: row for row in corpus_records}

---
# Stage 4 — Dense embedding index

**Input:** `corpus_records`. **Output:** `embedding_model`, `chunk_embeddings` (18 x 384).

Unchanged dense retrieval: `all-MiniLM-L6-v2`, normalised embeddings, cosine similarity
via dot product. The first run downloads the model (a few seconds in Colab).

In [ ]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME)

chunk_texts = [row["text"] for row in corpus_records]
chunk_embeddings = embedding_model.encode(
    chunk_texts,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True,
)

print("Index shape:", chunk_embeddings.shape)
assert chunk_embeddings.shape == (EXPECTED_CORPUS_SIZE, EXPECTED_EMBEDDING_DIM), (
    "Unexpected index shape - check the corpus and the embedding model."
)
print("Stage 4 PASS")

---
# Stage 5 — Query preprocessing utilities

**Input:** any string. **Output:** pure functions `light_stem`, `tokenize`,
`content_stems`, `extract_numbers`.

All four are deterministic and side-effect free, so they can be unit-tested on their own.
They are used by the reranker (Stage 6), the claim quality filter (Stage 12), and the
verifier (Stage 12).

In [ ]:
STOPWORDS = {
    "a", "an", "the", "and", "or", "of", "to", "for", "in", "on", "with",
    "is", "are", "was", "be", "been", "being", "should", "would", "could",
    "when", "what", "how", "who", "which", "that", "this", "those", "these",
    "if", "at", "by", "from", "as", "it", "their", "them", "they", "do",
    "does", "did", "not", "no", "yes", "than", "then", "into", "about",
    "adults", "adult", "people", "person", "type", "diabetes", "nice",
}

# Morphological families only (not chunk IDs). Helps guideline verbs/nouns
# that MiniLM often splits, e.g. intensified vs intensify.
STEM_ALIASES = {
    "intensified": "intensif",
    "intensify": "intensif",
    "intensifying": "intensif",
    "intensification": "intensif",
    "rise": "rise",
    "rises": "rise",
    "rising": "rise",
    "risen": "rise",
    "rose": "rise",
    "medicine": "medicin",
    "medicines": "medicin",
    "medication": "medicin",
    "medications": "medicin",
    "target": "target",
    "targets": "target",
    "offered": "offer",
    "offer": "offer",
    "offering": "offer",
    "measure": "measur",
    "measured": "measur",
    "measurement": "measur",
    "monitoring": "monitor",
    "monitor": "monitor",
    "contraindicated": "contraindic",
    "contraindication": "contraindic",
    "tolerated": "tolerat",
    "tolerance": "tolerat",
}


def light_stem(token: str) -> str:
    t = token.lower()
    if t in STEM_ALIASES:
        return STEM_ALIASES[t]
    for suf in ("ation", "ations", "ing", "ed", "es", "s", "ly"):
        if t.endswith(suf) and len(t) - len(suf) >= 4:
            t = t[:-len(suf)]
            break
    return STEM_ALIASES.get(t, t)


def tokenize(text: str) -> list[str]:
    text = text.lower().replace("%", " percent ")
    text = text.replace("mmol/mol", " mmol mol ")
    text = text.replace("hba1c", " hba1c ")
    text = text.replace("-", " ")
    # Keep lab-style tokens such as hba1c, sglt2, iscgm.
    return re.findall(r"[a-z]+\d*[a-z]*|\d+(?:\.\d+)?", text)


def content_stems(text: str) -> set[str]:
    return {light_stem(tok) for tok in tokenize(text) if tok not in STOPWORDS and len(tok) > 1}


def extract_numbers(text: str) -> set[str]:
    return set(re.findall(r"\d+(?:\.\d+)?", text.lower()))

In [ ]:
# --- Stage 5 self-test (independent of every other stage) -----------------
assert tokenize("HbA1c of 48 mmol/mol (6.5%)") == [
    "hba1c", "of", "48", "mmol", "mol", "6.5", "percent"
], tokenize("HbA1c of 48 mmol/mol (6.5%)")
assert light_stem("intensified") == light_stem("intensify") == "intensif"
assert light_stem("medications") == "medicin"
assert extract_numbers("48 mmol/mol (6.5%)") == {"48", "6.5"}
assert "diabetes" not in content_stems("adults with type 2 diabetes")  # stopword
print("Stage 5 PASS - preprocessing utilities behave as expected.")

---
# Stage 6 — Lexical and metadata reranking

**Input:** a question and one corpus record. **Output:** a dict of overlap features plus
a scalar `lexical_bonus`.

Dense cosine scores are never replaced. The bonus lifts chunks that share action stems,
numeric thresholds, or section/recommendation labels with the question. Weights come from
Stage 1 so they can be varied without editing this function.

In [ ]:
def lexical_bonus(question: str, row: dict[str, Any]) -> dict[str, float]:
    """Lexical/metadata overlap features for one (question, chunk) pair.

    Returns stem_overlap, number_overlap, meta_overlap and the combined
    lexical_bonus that is ADDED to the dense cosine score (never replaces it).
    """
    q_stems = content_stems(question)
    d_stems = content_stems(row["text"])
    meta_stems = content_stems(f"{row['section']} {row['recommendation_number']}")

    stem_overlap = 0.0
    if q_stems:
        stem_overlap = len(q_stems & d_stems) / len(q_stems)

    # Longer matched stems (intensify, metformin, hba1c) are more specific.
    distinctive = {s for s in (q_stems & d_stems) if len(s) >= DISTINCTIVE_STEM_MIN_LEN}
    distinctive_bonus = min(DISTINCTIVE_STEM_CAP, DISTINCTIVE_STEM_WEIGHT * len(distinctive))

    q_nums = extract_numbers(question)
    d_nums = extract_numbers(row["text"])
    number_overlap = 0.0
    if q_nums:
        number_overlap = len(q_nums & d_nums) / len(q_nums)

    meta_overlap = 0.0
    if q_stems and meta_stems:
        meta_overlap = len(q_stems & meta_stems) / len(q_stems)

    bonus = (
        STEM_OVERLAP_WEIGHT * stem_overlap
        + distinctive_bonus
        + NUMBER_OVERLAP_WEIGHT * number_overlap
        + META_OVERLAP_WEIGHT * meta_overlap
    )
    return {
        "stem_overlap": stem_overlap,
        "number_overlap": number_overlap,
        "meta_overlap": meta_overlap,
        "lexical_bonus": bonus,
    }


# --- Stage 6 self-test ----------------------------------------------------
_probe = lexical_bonus("When should treatment be intensified?", corpus_records[6])
assert 0.0 <= _probe["lexical_bonus"] <= 1.0
assert _probe["stem_overlap"] > 0.0
print("Stage 6 PASS - lexical_bonus returns bounded, non-zero overlap on a known pair.")
print(_probe)

---
# Stage 7 — Top-k retrieval function

**Input:** a query string and `k`. **Output:** a list of `k` hit dicts.

`final_score = cosine_similarity + lexical_bonus`. The retrieval methodology is unchanged
from the source notebook. The index, model, and corpus are passed as explicit default
arguments so the function can be tested against a different index if required.

In [ ]:
def retrieve_top_k(
    question: str,
    k: int = TOP_K,
    records: list[dict] | None = None,
    embeddings=None,
    model=None,
) -> list[dict[str, Any]]:
    """Dense cosine retrieval followed by the Stage 6 lexical/metadata rerank.

    Returns k hits, each carrying the evidence fields needed for citation:
    chunk_id, section, recommendation_number, page, source_url, and exact text.
    """
    records = corpus_records if records is None else records
    embeddings = chunk_embeddings if embeddings is None else embeddings
    model = embedding_model if model is None else model

    query_vec = model.encode(
        [question], convert_to_numpy=True, normalize_embeddings=True
    )[0]
    cosine = embeddings @ query_vec

    scored = []
    for i, row in enumerate(records):
        lex = lexical_bonus(question, row)
        scored.append({
            "index": i,
            "cosine": float(cosine[i]),
            "final_score": float(cosine[i]) + lex["lexical_bonus"],
            **lex,
        })

    scored.sort(key=lambda x: x["final_score"], reverse=True)

    hits = []
    for rank, item in enumerate(scored[:k], start=1):
        row = records[item["index"]]
        hits.append({
            "rank": rank,
            "chunk_id": row["chunk_id"],
            "section": row["section"],
            "recommendation_number": row["recommendation_number"],
            "page": row["page"],
            "cosine_similarity": item["cosine"],
            "lexical_bonus": item["lexical_bonus"],
            "similarity": item["final_score"],
            "text": row["text"],
            "source_url": row["source_url"],
        })
    return hits


# --- Stage 7 self-test ----------------------------------------------------
_hits = retrieve_top_k("What is the HbA1c target for type 2 diabetes?", k=TOP_K)
assert len(_hits) == TOP_K
assert [h["rank"] for h in _hits] == [1, 2, 3]
assert all(h["similarity"] >= _hits[-1]["similarity"] for h in _hits), "hits not sorted"
print("Stage 7 PASS - retrieve_top_k returns", TOP_K, "correctly ranked hits.")
print("Top-3 IDs:", [h["chunk_id"] for h in _hits])

---
# Stage 8 — Retrieval evaluation

**Input:** `retrieve_top_k`. **Output:** `smoke_test_score`, `retrieval_results`,
`hit_at_3_score`.

Two clearly separated question sets:

* **8a — smoke test (5 questions).** A fast sanity check only. Never reported as the
  evaluation result.
* **8b — main evaluation set (20 questions).** 15 answerable questions across all six
  corpus sections + 5 deliberately unsupported/out-of-scope questions. The unsupported
  questions are retained and reported separately; they have no gold chunk and are
  excluded from Hit@3.

The two sets live in **separate variables** (`SMOKE_TEST_QUESTIONS`, `EVAL_QUESTIONS`).
In the source notebook both were called `EVAL_QUESTIONS`, so whichever cell ran last
silently redefined the other — see the bug list.

### Stage 8a — Smoke test (5 questions, sanity check only)

In [ ]:
# SMOKE TEST ONLY - not the evaluation result. Question wording unchanged.
SMOKE_TEST_QUESTIONS = [
    {"question": "What is the HbA1c target for type 2 diabetes?",
     "expected_chunk_id": "NICE_NG28_006"},
    {"question": "What first-line medicines should be offered if there is no relevant comorbidity?",
     "expected_chunk_id": "NICE_NG28_008"},
    {"question": "When should treatment be intensified?",
     "expected_chunk_id": "NICE_NG28_007"},
    {"question": "What healthy eating advice should adults with type 2 diabetes follow?",
     "expected_chunk_id": "NICE_NG28_002"},
    {"question": "Should aspirin be offered if there is no cardiovascular disease?",
     "expected_chunk_id": "NICE_NG28_012"},
]

smoke_passed = 0
print("STAGE 8a - RETRIEVAL SMOKE TEST (5 questions, sanity check only)")
print("=" * 70)
for i, item in enumerate(SMOKE_TEST_QUESTIONS, start=1):
    ids = [h["chunk_id"] for h in retrieve_top_k(item["question"], k=TOP_K)]
    ok = item["expected_chunk_id"] in ids
    smoke_passed += int(ok)
    print(f"{i}. {'PASS' if ok else 'FAIL'} | gold={item['expected_chunk_id']} | top3={ids}")
    print(f"   Q: {item['question']}")

smoke_test_score = smoke_passed / len(SMOKE_TEST_QUESTIONS)
print("-" * 70)
print(f"Smoke test: {smoke_passed}/{len(SMOKE_TEST_QUESTIONS)} gold chunks in top {TOP_K}")
print("This is a sanity check, NOT the reported evaluation result.")

### Stage 8b — Main evaluation set (20 questions)

In [ ]:
# Manually curated 20-question evaluation set. Wording unchanged.
EVAL_QUESTIONS = [
    # ---- Answerable (15) — expected_chunk_id must exist in the corpus ----
    {"question": "What healthy eating advice should adults with type 2 diabetes follow?",
     "expected_chunk_id": "NICE_NG28_002", "category": "Dietary advice", "answerable": True},
    {"question": "What should be individualised when advising on carbohydrate and alcohol intake?",
     "expected_chunk_id": "NICE_NG28_003", "category": "Dietary advice", "answerable": True},
    {"question": "Should adults with type 2 diabetes use foods marketed specifically for people with diabetes?",
     "expected_chunk_id": "NICE_NG28_004", "category": "Dietary advice", "answerable": True},
    {"question": "How often should HbA1c levels be measured?",
     "expected_chunk_id": "NICE_NG28_005", "category": "Blood glucose management and HbA1c", "answerable": True},
    {"question": "What is the HbA1c target for type 2 diabetes?",
     "expected_chunk_id": "NICE_NG28_006", "category": "Blood glucose management and HbA1c", "answerable": True},
    {"question": "When should treatment be intensified?",
     "expected_chunk_id": "NICE_NG28_007", "category": "Blood glucose management and HbA1c", "answerable": True},
    {"question": "What first-line medicines should be offered if there is no relevant comorbidity?",
     "expected_chunk_id": "NICE_NG28_008", "category": "Initial medicines", "answerable": True},
    {"question": "What should be offered if metformin is contraindicated or not tolerated?",
     "expected_chunk_id": "NICE_NG28_009", "category": "Initial medicines", "answerable": True},
    {"question": "What treatment should be offered to adults with atherosclerotic cardiovascular disease?",
     "expected_chunk_id": "NICE_NG28_011", "category": "Cardiovascular risk", "answerable": True},
    {"question": "Should aspirin be offered if there is no cardiovascular disease?",
     "expected_chunk_id": "NICE_NG28_012", "category": "Cardiovascular risk", "answerable": True},
    {"question": "What should adults with type 2 diabetes be advised about periodontitis at their annual review?",
     "expected_chunk_id": "NICE_NG28_013", "category": "Complications", "answerable": True},
    {"question": "When should gastroparesis be considered as a diagnosis?",
     "expected_chunk_id": "NICE_NG28_014", "category": "Complications", "answerable": True},
    {"question": "When should adults be referred to eye screening after a type 2 diabetes diagnosis?",
     "expected_chunk_id": "NICE_NG28_016", "category": "Complications", "answerable": True},
    {"question": "When should self-monitoring of capillary blood glucose be offered?",
     "expected_chunk_id": "NICE_NG28_017", "category": "Monitoring", "answerable": True},
    {"question": "Who should be offered intermittently scanned continuous glucose monitoring (isCGM)?",
     "expected_chunk_id": "NICE_NG28_018", "category": "Monitoring", "answerable": True},

    # ---- Unsupported (5) — no gold chunk exists in this NG28 excerpt ----
    {"question": "What is the recommended target blood pressure for adults with type 2 diabetes?",
     "expected_chunk_id": None, "category": "Unsupported / Out-of-scope", "answerable": False},
    {"question": "What insulin dose adjustment is recommended during intercurrent illness (sick day rules)?",
     "expected_chunk_id": None, "category": "Unsupported / Out-of-scope", "answerable": False},
    {"question": "What statin or lipid-lowering therapy is recommended for adults with type 2 diabetes?",
     "expected_chunk_id": None, "category": "Unsupported / Out-of-scope", "answerable": False},
    {"question": "How should depression or psychological distress be screened for in adults with type 2 diabetes?",
     "expected_chunk_id": None, "category": "Unsupported / Out-of-scope", "answerable": False},
    {"question": "What is the target HbA1c during pregnancy for women with pre-existing type 2 diabetes?",
     "expected_chunk_id": None, "category": "Unsupported / Out-of-scope", "answerable": False},
]

n_answerable = sum(1 for q in EVAL_QUESTIONS if q["answerable"])
n_unsupported = sum(1 for q in EVAL_QUESTIONS if not q["answerable"])
print(f"Total questions: {len(EVAL_QUESTIONS)}")
print(f"Answerable:      {n_answerable}")
print(f"Unsupported:     {n_unsupported}")

# Guard against the two sets being confused for one another.
assert len(EVAL_QUESTIONS) == 20 and n_answerable == 15 and n_unsupported == 5
assert EVAL_QUESTIONS is not SMOKE_TEST_QUESTIONS
gold_ids = {q["expected_chunk_id"] for q in EVAL_QUESTIONS if q["answerable"]}
assert gold_ids <= set(chunk_lookup), "A gold chunk_id is not present in the corpus."
print("Stage 8b question set PASS")

### Stage 8c — Run retrieval and score Hit@3

In [ ]:
retrieval_results = []

for item in EVAL_QUESTIONS:
    hits = retrieve_top_k(item["question"], k=TOP_K)
    retrieved_ids = [h["chunk_id"] for h in hits]
    hit_at_3 = item["expected_chunk_id"] in retrieved_ids if item["answerable"] else None
    retrieval_results.append({
        "question": item["question"],
        "category": item["category"],
        "answerable": item["answerable"],
        "expected_chunk_id": item["expected_chunk_id"],
        "retrieved_chunk_ids": retrieved_ids,
        "hit_at_3": hit_at_3,
        "hits": hits,
    })

answerable_results = [r for r in retrieval_results if r["answerable"]]
unsupported_results = [r for r in retrieval_results if not r["answerable"]]

hits_count = sum(1 for r in answerable_results if r["hit_at_3"])
hit_at_3_score = hits_count / len(answerable_results) if answerable_results else 0.0

print("STAGE 8c - Hit@3 (answerable questions only)")
print("=" * 70)
for r in answerable_results:
    status = "HIT " if r["hit_at_3"] else "MISS"
    print(f"[{status}] gold={r['expected_chunk_id']:<16} top3={r['retrieved_chunk_ids']}")
    print(f"        Q: {r['question']}")
print("-" * 70)
print(f"Hit@{TOP_K} = {hits_count}/{len(answerable_results)} = {hit_at_3_score:.2%}")

print()
print("Unsupported / out-of-scope questions (retained, reported separately, not scored)")
print("=" * 70)
for r in unsupported_results:
    print(f"Q: {r['question']}")
    print(f"   category: {r['category']}")
    print(f"   retriever still returned (inspection only): {r['retrieved_chunk_ids']}")
print(f"Total unsupported questions: {len(unsupported_results)}")

---
# Stage 9 — Evidence display and manual reference records

**Input:** `retrieval_results`. **Output:** printed evidence, `manual_claim_records`,
`manual_claim_records.csv` / `.json`.

Two distinct things live here, and they are kept distinct:

1. `display_evidence_for_review()` — prints `chunk_id`, `recommendation_number`, `page`,
   `exact_text`, and `source_url` for every hit, for every question.
2. `manual_claim_records` — the **human-annotated gold/reference set** (29 rows).
   These are written by the annotator, not by any model. `recommendation_number`,
   `page`, and `source_url` are looked up from the corpus rather than retyped.
   Automated output never overwrites this variable or these files.

In [ ]:
def display_evidence_for_review(result: dict) -> None:
    """Print the full citable evidence for one evaluation question."""
    print("=" * 88)
    print(f"QUESTION: {result['question']}")
    print(f"category: {result['category']}  |  answerable: {result['answerable']}  "
          f"|  expected_chunk_id: {result['expected_chunk_id']}")
    print("=" * 88)
    for hit in result["hits"]:
        print(f"\n--- Rank {hit['rank']} ---")
        print(f"chunk_id:               {hit['chunk_id']}")
        print(f"recommendation_number:  {hit['recommendation_number']}")
        print(f"page:                   {hit['page']}")
        print(f"cosine_similarity:      {hit['cosine_similarity']:.4f}")
        print(f"lexical_bonus:          {hit['lexical_bonus']:.4f}")
        print(f"similarity_score:       {hit['similarity']:.4f}")
        print(f"source_url:             {hit['source_url']}")
        print("exact_text:")
        print(hit["text"])
        print("-" * 88)
    print()


SHOW_ALL_EVIDENCE = True  # set False to keep the notebook output short

if SHOW_ALL_EVIDENCE:
    for r in retrieval_results:
        display_evidence_for_review(r)
else:
    display_evidence_for_review(retrieval_results[0])
    print(f"({len(retrieval_results) - 1} further questions suppressed; "
          f"set SHOW_ALL_EVIDENCE = True to print them.)")

### Stage 9b — Manually curated gold/reference claim records (human-written)

In [ ]:
# >>> MANUAL ANNOTATION - human gold labels. Do not overwrite from any model. <<<
MANUAL_ANSWERABLE_CLAIM_DATA = {
    "What healthy eating advice should adults with type 2 diabetes follow?": {
        "manual_answer": "Adults with type 2 diabetes should follow the same healthy eating "
            "advice as the general population, including high-fibre, low-glycaemic-index "
            "carbohydrates, low-fat dairy, oily fish, and limiting saturated and trans fat intake.",
        "claims": [
            ("Adults with type 2 diabetes should be encouraged to follow the same healthy "
             "eating advice as the general population.", "NICE_NG28_002", "Supported"),
            ("This includes eating high-fibre, low-glycaemic-index carbohydrate sources such "
             "as fruit, vegetables, wholegrains and pulses; choosing low-fat dairy products; "
             "eating oily fish; and controlling saturated and trans fatty acid intake.",
             "NICE_NG28_002", "Supported"),
        ],
    },
    "What should be individualised when advising on carbohydrate and alcohol intake?": {
        "manual_answer": "Recommendations for carbohydrate and alcohol intake and meal patterns "
            "should be individualised, with reducing hypoglycaemia risk a particular aim for "
            "people using insulin or an insulin secretagogue.",
        "claims": [
            ("Recommendations for carbohydrate and alcohol intake, and meal patterns, should "
             "be individualised.", "NICE_NG28_003", "Supported"),
            ("Reducing the risk of hypoglycaemia should be a particular aim for people using "
             "insulin or an insulin secretagogue.", "NICE_NG28_003", "Supported"),
        ],
    },
    "Should adults with type 2 diabetes use foods marketed specifically for people with diabetes?": {
        "manual_answer": "No — adults with type 2 diabetes should be discouraged from using "
            "foods marketed specifically for people with diabetes.",
        "claims": [
            ("Adults with type 2 diabetes should be discouraged from using foods marketed "
             "specifically for people with diabetes.", "NICE_NG28_004", "Supported"),
        ],
    },
    "How often should HbA1c levels be measured?": {
        "manual_answer": "HbA1c should be measured every 3 to 6 months (tailored to individual "
            "needs) until it is stable on unchanging therapy, then every 6 months once HbA1c "
            "and blood glucose lowering therapy are stable.",
        "claims": [
            ("HbA1c levels should be measured every 3 to 6 months, tailored to individual "
             "needs, until HbA1c is stable on unchanging therapy.", "NICE_NG28_005", "Supported"),
            ("Once the HbA1c level and blood glucose lowering therapy are stable, HbA1c "
             "should be measured every 6 months.", "NICE_NG28_005", "Supported"),
        ],
    },
    "What is the HbA1c target for type 2 diabetes?": {
        "manual_answer": "Adults managed by healthy living and diet alone, or diet combined "
            "with an initial medication not associated with hypoglycaemia, should aim for an "
            "HbA1c of 48 mmol/mol (6.5%). Adults on a medicine associated with hypoglycaemia "
            "should aim for 53 mmol/mol (7.0%).",
        "claims": [
            ("Adults managed by healthy living and diet, or diet combined with an initial "
             "medication regimen not associated with hypoglycaemia, should be supported to "
             "aim for an HbA1c level of 48 mmol/mol (6.5%).", "NICE_NG28_006", "Supported"),
            ("Adults on a medicine associated with hypoglycaemia should be supported to aim "
             "for an HbA1c level of 53 mmol/mol (7.0%).", "NICE_NG28_006", "Supported"),
        ],
    },
    "When should treatment be intensified?": {
        "manual_answer": "Treatment should be intensified if HbA1c is not adequately "
            "controlled by the initial medication regimen and rises to 58 mmol/mol (7.5%) or "
            "higher; diet and medicine-adherence advice should be reinforced and the person "
            "supported toward an HbA1c of 53 mmol/mol (7.0%).",
        "claims": [
            ("Treatment should be intensified if HbA1c is not adequately controlled by the "
             "initial medication regimen and rises to 58 mmol/mol (7.5%) or higher.",
             "NICE_NG28_007", "Supported"),
            ("In that situation, diet, healthy living, and medicine adherence advice should "
             "be reinforced and the person supported to aim for an HbA1c level of 53 "
             "mmol/mol (7.0%).", "NICE_NG28_007", "Supported"),
        ],
    },
    "What first-line medicines should be offered if there is no relevant comorbidity?": {
        "manual_answer": "Adults with type 2 diabetes and no relevant comorbidity should be "
            "offered modified-release metformin and an SGLT-2 inhibitor.",
        "claims": [
            ("Adults with type 2 diabetes and no relevant comorbidity should be offered "
             "modified-release metformin and an SGLT-2 inhibitor.", "NICE_NG28_008", "Supported"),
        ],
    },
    "What should be offered if metformin is contraindicated or not tolerated?": {
        "manual_answer": "If metformin is contraindicated or not tolerated, monotherapy with "
            "an SGLT-2 inhibitor should be offered.",
        "claims": [
            ("If metformin is contraindicated or not tolerated, monotherapy with an SGLT-2 "
             "inhibitor should be offered.", "NICE_NG28_009", "Supported"),
        ],
    },
    "What treatment should be offered to adults with atherosclerotic cardiovascular disease?": {
        "manual_answer": "Adults with type 2 diabetes and atherosclerotic cardiovascular "
            "disease should be offered modified-release metformin, an SGLT-2 inhibitor, and "
            "subcutaneous semaglutide (Ozempic) up to 1 mg once a week for its cardiovascular, "
            "renal and glycaemic benefits.",
        "claims": [
            ("Adults with type 2 diabetes and atherosclerotic cardiovascular disease should "
             "be offered modified-release metformin and an SGLT-2 inhibitor.",
             "NICE_NG28_011", "Supported"),
            ("They should also be offered subcutaneous semaglutide (Ozempic), up to 1 mg "
             "once a week, for its cardiovascular, renal and glycaemic benefits.",
             "NICE_NG28_011", "Supported"),
        ],
    },
    "Should aspirin be offered if there is no cardiovascular disease?": {
        "manual_answer": "No — antiplatelet therapy (aspirin or clopidogrel) should not be "
            "offered to adults with type 2 diabetes who do not have cardiovascular disease.",
        "claims": [
            ("Antiplatelet therapy (aspirin or clopidogrel) should not be offered to adults "
             "with type 2 diabetes without cardiovascular disease.", "NICE_NG28_012", "Supported"),
        ],
    },
    "What should adults with type 2 diabetes be advised about periodontitis at their annual review?": {
        "manual_answer": "At their annual review, adults with type 2 diabetes should be "
            "advised that they are at higher risk of periodontitis, and that managing "
            "periodontitis, if they get it, can improve blood glucose control and reduce the "
            "risk of hyperglycaemia.",
        "claims": [
            ("At their annual review, adults with type 2 diabetes should be advised that "
             "they are at higher risk of periodontitis.", "NICE_NG28_013", "Supported"),
            ("They should be advised that managing periodontitis, if they get it, can "
             "improve their blood glucose control and reduce their risk of hyperglycaemia.",
             "NICE_NG28_013", "Supported"),
        ],
    },
    "When should gastroparesis be considered as a diagnosis?": {
        "manual_answer": "Gastroparesis should be considered as a diagnosis in adults with "
            "type 2 diabetes who have erratic blood glucose control or unexplained gastric "
            "bloating or vomiting, taking into account possible alternative diagnoses.",
        "claims": [
            ("Gastroparesis should be considered in adults with type 2 diabetes who have "
             "erratic blood glucose control or unexplained gastric bloating or vomiting, "
             "taking into account possible alternative diagnoses.", "NICE_NG28_014", "Supported"),
        ],
    },
    "When should adults be referred to eye screening after a type 2 diabetes diagnosis?": {
        "manual_answer": "Adults should be referred immediately to the local eye screening "
            "service when they are diagnosed with type 2 diabetes.",
        "claims": [
            ("Adults should be referred immediately to the local eye screening service when "
             "they are diagnosed with type 2 diabetes.", "NICE_NG28_016", "Supported"),
        ],
    },
    "When should self-monitoring of capillary blood glucose be offered?": {
        "manual_answer": "Self-monitoring of capillary blood glucose should not be routinely "
            "offered to adults with type 2 diabetes, unless they are on insulin, have "
            "evidence of hypoglycaemic episodes, are on oral medication that may increase "
            "hypoglycaemia risk while driving or operating machinery, or are pregnant or "
            "planning pregnancy.",
        "claims": [
            ("Self-monitoring of capillary blood glucose should not be routinely offered to "
             "adults with type 2 diabetes.", "NICE_NG28_017", "Supported"),
            ("It should be offered if the person is on insulin, has evidence of "
             "hypoglycaemic episodes, is on oral medication that may increase their risk of "
             "hypoglycaemia while driving or operating machinery, or is pregnant or planning "
             "to become pregnant.", "NICE_NG28_017", "Supported"),
        ],
    },
    "Who should be offered intermittently scanned continuous glucose monitoring (isCGM)?": {
        "manual_answer": "isCGM ('flash') should be offered to adults with type 2 diabetes on "
            "multiple daily insulin injections who have recurrent or severe hypoglycaemia, "
            "impaired hypoglycaemia awareness, a condition or disability that prevents "
            "capillary self-monitoring but allows isCGM use, or would otherwise need to "
            "self-measure at least 8 times a day.",
        "claims": [
            ("isCGM (commonly referred to as 'flash') should be offered to adults with type "
             "2 diabetes on multiple daily insulin injections if certain criteria apply.",
             "NICE_NG28_018", "Supported"),
            ("Criteria include recurrent or severe hypoglycaemia, impaired hypoglycaemia "
             "awareness, a condition or disability that prevents capillary blood glucose "
             "self-monitoring but allows use of an isCGM device, or otherwise needing to "
             "self-measure at least 8 times a day.", "NICE_NG28_018", "Supported"),
        ],
    },
}


def build_manual_claim_record(question, category, manual_answer, atomic_claim, supporting_chunk_id, status):
    chunk = chunk_lookup.get(supporting_chunk_id) if supporting_chunk_id else None
    return {
        "question": question,
        "category": category,
        "manual_answer": manual_answer,
        "atomic_claim": atomic_claim,
        "supporting_chunk_id": supporting_chunk_id,
        "recommendation_number": chunk["recommendation_number"] if chunk else None,
        "page": chunk["page"] if chunk else None,
        "source_url": chunk["source_url"] if chunk else None,
        "status": status,
    }


manual_claim_records = []

for item in EVAL_QUESTIONS:
    if item["answerable"]:
        data = MANUAL_ANSWERABLE_CLAIM_DATA[item["question"]]
        for atomic_claim, supporting_chunk_id, status in data["claims"]:
            manual_claim_records.append(
                build_manual_claim_record(
                    question=item["question"],
                    category=item["category"],
                    manual_answer=data["manual_answer"],
                    atomic_claim=atomic_claim,
                    supporting_chunk_id=supporting_chunk_id,
                    status=status,
                )
            )
    else:
        manual_claim_records.append(
            build_manual_claim_record(
                question=item["question"],
                category=item["category"],
                manual_answer="The retrieved evidence is insufficient to answer this question.",
                atomic_claim="The corpus does not contain sufficient evidence for this question.",
                supporting_chunk_id=None,
                status="Insufficient Evidence",
            )
        )

print(f"{len(manual_claim_records)} claim record(s) built "
      f"({len(MANUAL_ANSWERABLE_CLAIM_DATA)} answerable questions, "
      f"{sum(1 for r in manual_claim_records if r['status'] == 'Insufficient Evidence' and r['supporting_chunk_id'] is None)} "
      f"unsupported-question rows).")

In [ ]:
# --- Validation of the manual gold records (unchanged logic) --------------
valid_chunk_ids = set(chunk_lookup)
question_answerable_lookup = {item["question"]: item["answerable"] for item in EVAL_QUESTIONS}

manual_validation_warnings = []

for i, rec in enumerate(manual_claim_records):
    if rec["status"] not in ALLOWED_STATUSES:
        manual_validation_warnings.append(
            f"Row {i} ({rec['question'][:60]!r}): status {rec['status']!r} is not a valid final status."
        )
    if rec["supporting_chunk_id"] is not None and rec["supporting_chunk_id"] not in valid_chunk_ids:
        manual_validation_warnings.append(
            f"Row {i}: supporting_chunk_id {rec['supporting_chunk_id']!r} does not exist in the corpus."
        )
    if not question_answerable_lookup.get(rec["question"], True) and rec["supporting_chunk_id"] is not None:
        manual_validation_warnings.append(
            f"Row {i}: unsupported question but supporting_chunk_id is not None."
        )
    if "EDIT ME" in str(rec.get("manual_answer", "")) or "EDIT ME" in str(rec.get("atomic_claim", "")):
        manual_validation_warnings.append(f"Row {i}: placeholder 'EDIT ME' text still present.")

print(f"Manual gold records: {len(manual_claim_records)}")
print(f"Validation: {len(manual_validation_warnings)} warning(s)")
for w in manual_validation_warnings:
    print(" -", w)
if not manual_validation_warnings:
    print("All manual records passed validation.")

# --- Export (manual set only) --------------------------------------------
import pandas as pd

manual_claim_df = pd.DataFrame(manual_claim_records)
manual_claim_df.to_csv(MANUAL_CLAIMS_CSV, index=False)
with open(MANUAL_CLAIMS_JSON, "w") as f:
    json.dump(manual_claim_records, f, indent=2)

print(f"\nSaved {len(manual_claim_df)} MANUAL (human gold) rows to "
      f"{MANUAL_CLAIMS_CSV} and {MANUAL_CLAIMS_JSON}")
manual_claim_df.head()

---
# Stage 10 — Extractive QA answer candidates

**Input:** `retrieval_results`. **Output:** `qa_candidate_answers`.

Local `distilbert-base-cased-distilled-squad` loaded directly with `AutoTokenizer` +
`AutoModelForQuestionAnswering`. No `pipeline()` call, no API key, no hosted service.

The earlier `pipeline("question-answering", ...)` implementation failed in this
environment (the task name was not registered by the installed `transformers` build) and
has been removed from the working notebook — see the change log.

A candidate span is **not** a final clinical answer: it is a short extractive span pulled
from a retrieved chunk. Unsupported questions are not sent to the QA model; they receive a
fixed placeholder row.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForQuestionAnswering

qa_model_loaded = False
qa_tokenizer = None
qa_model = None
qa_load_error = None

try:
    qa_tokenizer = AutoTokenizer.from_pretrained(QA_MODEL_NAME)
    qa_model = AutoModelForQuestionAnswering.from_pretrained(QA_MODEL_NAME)
    qa_model.eval()
    qa_model_loaded = True
    print(f"QA model loaded: True  ({QA_MODEL_NAME})")
except Exception as exc:  # no internet, OOM, etc. -- do not crash the notebook
    qa_load_error = str(exc)
    print("QA model loaded: False")
    print(f"Load error (Stage 10 will be skipped): {qa_load_error}")

In [ ]:
def extract_answer_span(
    question: str,
    context: str,
    max_answer_len: int = QA_MAX_ANSWER_LEN,
    top_k: int = QA_TOP_K_LOGITS,
) -> tuple[str, float]:
    """Direct local extractive QA using AutoModelForQuestionAnswering.

    Tokenizes (question, context) as a pair, restricts the span search to context
    tokens only (question tokens excluded via sequence_ids), and returns
    (answer_text, confidence). Returns ("", 0.0) when no valid span is found,
    when [CLS] wins (the model's own "no answer here" signal), or when the model
    is not loaded.
    """
    if not qa_model_loaded:
        return "", 0.0

    inputs = qa_tokenizer(
        question,
        context,
        return_tensors="pt",
        truncation="only_second",
        max_length=QA_MAX_SEQ_LEN,
    )

    with torch.no_grad():
        outputs = qa_model(**inputs)

    start_logits = outputs.start_logits[0]
    end_logits = outputs.end_logits[0]

    sequence_ids = inputs.sequence_ids(0)
    context_token_indices = [i for i, sid in enumerate(sequence_ids) if sid == 1]
    if not context_token_indices:
        return "", 0.0

    ctx_start, ctx_end = context_token_indices[0], context_token_indices[-1]

    k = min(top_k, ctx_end - ctx_start + 1)
    start_candidates = (torch.topk(start_logits[ctx_start:ctx_end + 1], k=k).indices + ctx_start).tolist()
    end_candidates = (torch.topk(end_logits[ctx_start:ctx_end + 1], k=k).indices + ctx_start).tolist()

    best_score = float("-inf")
    best_start, best_end = None, None

    for s in start_candidates:
        for e in end_candidates:
            if e < s or (e - s + 1) > max_answer_len:
                continue
            score = (start_logits[s] + end_logits[e]).item()
            if score > best_score:
                best_score = score
                best_start, best_end = s, e

    if best_start is None or best_start == 0:
        return "", 0.0

    input_ids = inputs["input_ids"][0]
    answer_text = qa_tokenizer.decode(
        input_ids[best_start:best_end + 1], skip_special_tokens=True
    ).strip()
    if not answer_text:
        return "", 0.0

    start_probs = torch.softmax(start_logits, dim=0)
    end_probs = torch.softmax(end_logits, dim=0)
    confidence = round(float((start_probs[best_start] * end_probs[best_end]).item()), 4)

    return answer_text, confidence


# --- Stage 10 self-test ---------------------------------------------------
if qa_model_loaded:
    _span, _conf = extract_answer_span(
        "What is the HbA1c target?", chunk_lookup["NICE_NG28_006"]["text"]
    )
    assert isinstance(_span, str) and isinstance(_conf, float)
    print(f"Self-test span: {_span!r} (confidence {_conf})")
    print("Stage 10 function PASS")

In [ ]:
NO_EVIDENCE_TEXT = "The retrieved evidence is insufficient to answer this question."

qa_candidate_answers = []  # one row per (question, chunk) candidate span

if qa_model_loaded:
    for r in retrieval_results:
        if not r["answerable"]:
            qa_candidate_answers.append({
                "question": r["question"],
                "category": r["category"],
                "answerable": False,
                "chunk_id": None,
                "candidate_span": NO_EVIDENCE_TEXT,
                "confidence": None,
            })
            continue

        for hit in r["hits"]:
            span, score = extract_answer_span(r["question"], hit["text"])
            qa_candidate_answers.append({
                "question": r["question"],
                "category": r["category"],
                "answerable": True,
                "chunk_id": hit["chunk_id"],
                "candidate_span": span,
                "confidence": score if span else None,
            })

    print(f"Stage 10: generated {len(qa_candidate_answers)} candidate rows "
          f"(including unsupported-question placeholders).")
else:
    print("Stage 10 skipped: QA model not loaded.")

In [ ]:
# --- Stage 10 output and PASS/FAIL ---------------------------------------
if qa_model_loaded:
    print("=" * 100)
    print("STAGE 10 - AUTOMATED EXTRACTIVE ANSWER CANDIDATES")
    print("(short spans from retrieved context - NOT final clinical answers)")
    print("=" * 100)

    current_question = None
    for row in qa_candidate_answers:
        if row["question"] != current_question:
            current_question = row["question"]
            print(f"\nQUESTION: {current_question}")
        if not row["answerable"]:
            print(f"  [unsupported] {row['candidate_span']}")
            continue
        span = row["candidate_span"] or "(no span returned)"
        print(f"  chunk {row['chunk_id']} | conf={row['confidence']} | span: {span}")

    answerable_questions = [q["question"] for q in EVAL_QUESTIONS if q["answerable"]]
    with_span = {
        row["question"] for row in qa_candidate_answers
        if row["answerable"] and row["candidate_span"]
    }
    n_with_span = len(with_span)
    unsupported_rows_ok = all(
        row["candidate_span"] == NO_EVIDENCE_TEXT
        for row in qa_candidate_answers if not row["answerable"]
    )
    stage10_status = (
        "PASS" if n_with_span == len(answerable_questions) and unsupported_rows_ok else "FAIL"
    )

    print("\n" + "=" * 60)
    print("STAGE 10 SUMMARY")
    print("=" * 60)
    print(f"Model loaded:                       {qa_model_loaded}")
    print(f"Answerable questions with a span:   {n_with_span}/{len(answerable_questions)}")
    print(f"Unsupported rows correctly held:    {unsupported_rows_ok}")
    print(f"Stage 10 status:                    {stage10_status}")
    print("Limitation: extractive spans only. The model copies text from the retrieved")
    print("chunk; it does not reason about clinical correctness or completeness.")
    print("=" * 60)
else:
    stage10_status = "SKIPPED"
    print("Stage 10 status: SKIPPED (model not loaded).")

---
# Stage 11 — Rule-based claim decomposition

**Input:** `qa_candidate_answers`. **Output:** `atomic_claims`.

`split_into_atomic_claims()` is **rule-based and LLM-free**. It only cuts the original
span at a delimiter; it never paraphrases, invents, reorders, or rewrites text. Splits
occur on (1) semicolons, (2) a clear `" and "` conjunction where both halves are
substantial clauses, and (3) two distinct `<value> mmol/mol (<value>%)` numeric clauses.

In [ ]:
NUMERIC_PAIR_PATTERN = re.compile(
    r"\d+(?:\.\d+)?\s*mmol/mol\s*\(\d+(?:\.\d+)?%\)", re.IGNORECASE
)


def split_into_atomic_claims(span: str) -> list[str]:
    """
    Rule-based, LLM-free claim splitter. Splits only on:
      1. semicolons;
      2. a clear \' and \' conjunction where BOTH resulting halves are
         substantial clauses (>= 3 words each);
      3. two clearly separate numeric clauses, e.g. two distinct
         "<value> mmol/mol (<value>%)" thresholds in the same segment.
    Never paraphrases, invents, or reorders text -- only cuts the original
    span at a delimiter.
    """
    span = span.strip()
    if not span:
        return []

    segments = [s.strip() for s in span.split(";") if s.strip()]
    final_segments = []

    for seg in segments:
        numeric_matches = list(NUMERIC_PAIR_PATTERN.finditer(seg))

        if len(numeric_matches) >= 2:
            split_point = numeric_matches[1].start()
            left = seg[:split_point].rstrip(" ,").rstrip()
            if left.lower().endswith(" and"):
                left = left[: -len(" and")].rstrip()
            right = seg[split_point:].strip()
            if len(left.split()) >= 3 and len(right.split()) >= 3:
                final_segments.extend([left, right])
                continue

        and_split = re.split(r"\s+and\s+", seg)
        if len(and_split) == 2 and all(len(p.split()) >= 3 for p in and_split):
            final_segments.extend(p.strip() for p in and_split)
        else:
            final_segments.append(seg)

    return [f for f in final_segments if f]


# --- Stage 11 self-test ---------------------------------------------------
assert split_into_atomic_claims("") == []
assert split_into_atomic_claims("a single clause") == ["a single clause"]
assert len(split_into_atomic_claims("measure HbA1c regularly; review the treatment plan")) == 2
assert split_into_atomic_claims("bread and butter") == ["bread and butter"]  # halves too short
print("Stage 11 splitter PASS")

In [ ]:
atomic_claims = []

for row in qa_candidate_answers:
    if not row["answerable"]:
        atomic_claims.append({
            "question": row["question"],
            "category": row["category"],
            "original_extracted_span": None,
            "atomic_claim": NO_EVIDENCE_TEXT,
            "source_chunk_id": None,
            "qa_confidence": None,
        })
        continue

    span = row["candidate_span"]
    if not span:
        # No valid Stage 10 span for this chunk -- nothing to decompose.
        continue

    for claim in split_into_atomic_claims(span):
        atomic_claims.append({
            "question": row["question"],
            "category": row["category"],
            "original_extracted_span": span,
            "atomic_claim": claim,
            "source_chunk_id": row["chunk_id"],
            "qa_confidence": row["confidence"],
        })

answerable_claim_rows = [r for r in atomic_claims if r["source_chunk_id"] is not None]
placeholder_rows = [r for r in atomic_claims if r["source_chunk_id"] is None]

print(f"Stage 11: produced {len(atomic_claims)} row(s) "
      f"({len(answerable_claim_rows)} atomic claims + "
      f"{len(placeholder_rows)} unsupported-question placeholders).")

print("\n" + "=" * 100)
print("SAMPLE OF DECOMPOSED CLAIMS (rule-based, no LLM)")
print("=" * 100)
for row in answerable_claim_rows[:10]:
    print(f"\nQUESTION: {row['question']}")
    print(f"  original_extracted_span: {row['original_extracted_span']}")
    print(f"  atomic_claim:            {row['atomic_claim']}")
    print(f"  source_chunk_id:         {row['source_chunk_id']}")

# --- Stage 11 PASS/FAIL ---------------------------------------------------
n_spans = sum(1 for r in qa_candidate_answers if r["answerable"] and r["candidate_span"])
no_invented_text = all(
    r["atomic_claim"] in r["original_extracted_span"] or
    r["atomic_claim"].rstrip(".") in r["original_extracted_span"]
    for r in answerable_claim_rows
)
if not qa_model_loaded:
    stage11_status = "SKIPPED"  # no Stage 10 spans available; not a failure of this stage
else:
    stage11_status = (
        "PASS" if len(answerable_claim_rows) >= n_spans and
        len(placeholder_rows) == len(unsupported_results) and no_invented_text else "FAIL"
    )
print("\n" + "=" * 60)
print("STAGE 11 SUMMARY")
print("=" * 60)
print(f"Stage 10 spans decomposed:      {n_spans}")
print(f"Atomic claims produced:         {len(answerable_claim_rows)}")
print(f"Placeholder rows:               {len(placeholder_rows)}")
print(f"No text invented or rewritten:  {no_invented_text}")
print(f"Stage 11 status:                {stage11_status}")
print("=" * 60)

---
# Stage 12 — Claim verification and final report

**Input:** `atomic_claims`. **Output:** `claim_verification_records`,
`automated_claim_verification.csv` / `.json`, `final_report.md`, `final_summary.json`.

### What this verifier is, and what it is not

The verifier uses **only three signals**:

1. the retrieval **similarity score** of the top-1 chunk retrieved for the claim text,
2. **lexical overlap** between the claim's content stems and the evidence's content stems,
3. **numerical consistency** between numbers in the claim and numbers in the evidence.

It does **not** perform natural-language inference. It has no model of entailment,
contradiction, negation, or clinical validity. A `Supported` label means *"the retrieved
guideline text is lexically and numerically consistent with this claim"*, not *"this claim
is clinically true"*. `Unsupported` is reserved for an explicit numeric conflict.

A quality filter runs **before** verification, so incomplete extractive fragments
(`"diet"`, `"2015, amended 2026"`, dangling conditionals) are never labelled `Supported`.
Short but clinically specific spans (a threshold, a drug name, a dosing interval) are
exempted from the fragment rule.

This is the final retained version of the verifier. Three earlier iterations existed in the
source notebook; see the change log.

### Stage 12a — Claim quality filter

In [ ]:
GENERIC_FRAGMENT_STOPLIST = {
    "diet", "renal", "pregnancy", "recommendations", "recommendation", "nutritional",
}

DANGLING_STARTERS = {
    "if", "when", "unless", "although", "while", "since", "because",
}

DATE_VERSION_KEYWORDS = {
    "amended", "revised", "updated", "version", "edition", "reviewed", "published",
}

SPECIFIC_CONTENT_PATTERNS = [
    r"\d+(?:\.\d+)?\s*mmol/mol",   # e.g. "48 mmol/mol"
    r"\d+(?:\.\d+)?\s*%",          # e.g. "7.0%"
    r"\d+\s*times?\s*a\s*day",     # e.g. "8 times a day"
    r"\bat least\b",
    r"\bmonotherapy\b",
    r"\bmodified-release\b",
    r"\bmultiple daily\b",
    r"\binsulin\b",
    r"\bmetformin\b",
    r"\bsglt-?2\b",
    r"\bdpp-?4\b",
    r"\bglp-?1\b",
    r"\bsulfonylurea\b",
    r"\bpioglitazone\b",
    r"\binjections?\b",
    r"\binhibitor\b",
]

MONITORING_VERB_STEMS = {"assess", "monitor", "review", "measur", "check", "screen"}


def is_pure_date_or_version(claim: str) -> bool:
    """True if every token is a bare year/number or a date/version keyword,
    e.g. "2015, amended 2026"."""
    raw_tokens = [t for t in claim.lower().replace(",", " ").split() if t]
    if not raw_tokens:
        return False
    for tok in raw_tokens:
        if tok in DATE_VERSION_KEYWORDS:
            continue
        if re.fullmatch(r"\d{4}", tok):
            continue
        if re.fullmatch(r"v?\d+(?:\.\d+)*", tok):
            continue
        return False
    return True


def matches_specific_content(claim_lower: str) -> bool:
    return any(re.search(pattern, claim_lower) for pattern in SPECIFIC_CONTENT_PATTERNS)


def classify_claim_quality(claim: str) -> tuple[str, str]:
    """Completeness filter, run BEFORE any evidence check.
    Returns (quality, reason) where quality is "Valid" or "Fragment".
    Short but clinically specific spans (threshold, drug name, dosing interval)
    are exempted from the content-token minimum."""
    stripped = claim.strip()
    tokens = stripped.split()
    lower = stripped.lower()

    if len(tokens) <= 1:
        return "Fragment", "One-word claim -- not a checkable proposition."

    if is_pure_date_or_version(stripped):
        return "Fragment", "Pure date/version string with no substantive content."

    if lower.rstrip(".") in GENERIC_FRAGMENT_STOPLIST:
        return "Fragment", "Generic single-concept fragment with no complete proposition."

    if matches_specific_content(lower):
        return "Valid", (
            "Specific treatment/threshold/time-interval/eligibility content detected "
            "(numeric threshold, drug or drug-class name, or dosing interval); "
            "treated as a direct short-answer span."
        )

    stems = content_stems(stripped)
    if len(stems) < CLAIM_MIN_CONTENT_STEMS:
        return "Fragment", (
            f"Only {len(stems)} meaningful content token(s) -- below the "
            f"{CLAIM_MIN_CONTENT_STEMS}-token completeness threshold, and no specific "
            f"treatment/threshold/eligibility pattern detected."
        )

    first_word = tokens[0].lower().strip(",.")
    has_attached_outcome = ("," in stripped) or (" then " in lower)
    if first_word in DANGLING_STARTERS and not has_attached_outcome:
        return "Fragment", (
            "Dangling conditional/subordinate clause with no attached outcome; "
            "incomplete proposition on its own."
        )

    return "Valid", "Passes completeness checks: multi-word, substantive, no dangling conditional."


def check_question_context_match(question: str, claim: str) -> tuple[bool, str]:
    """Narrow heuristic; defaults to True. Flips to False only when the claim opens
    with a monitoring/assessment verb the question does not ask about."""
    claim_tokens = tokenize(claim)
    if not claim_tokens:
        return True, "No claim tokens to check; default compatible."

    leading_stem = light_stem(claim_tokens[0])
    question_stems = content_stems(question)

    if leading_stem in MONITORING_VERB_STEMS and leading_stem not in question_stems:
        return False, (
            f'Claim opens with a monitoring/assessment action ("{claim_tokens[0]}") that '
            f"the question does not ask about; likely a different clinical action than asked."
        )

    return True, "No topic-mismatch signal detected (narrow heuristic; default compatible)."


# --- Stage 12a self-test --------------------------------------------------
assert classify_claim_quality("diet")[0] == "Fragment"                 # stoplist
assert classify_claim_quality("2015, amended 2026")[0] == "Fragment"   # date/version
assert classify_claim_quality("When the level rises")[0] == "Fragment" # too few content stems
assert classify_claim_quality("Although the level rises further")[0] == "Fragment"  # dangling
assert classify_claim_quality("48 mmol/mol")[0] == "Valid"             # specific-content override
assert classify_claim_quality("modified-release metformin")[0] == "Valid"
assert check_question_context_match("What is the HbA1c target?", "Monitor renal function")[0] is False

# KNOWN DEFECT, pinned rather than silently patched (see bug list, item C4):
# light_stem("assess") -> "asses" (the trailing "s" is stripped), so the "assess"
# entry in MONITORING_VERB_STEMS can never be produced by the stemmer and
# claims opening with "Assess ..." are NOT flagged as context mismatches.
assert light_stem("assess") == "asses"
assert check_question_context_match("What is the HbA1c target?", "Assess renal function")[0] is True

# DOCUMENTED BEHAVIOUR (not a silent change): the specific-content override is
# evaluated BEFORE the dangling-conditional rule, so a dangling conditional that
# names a drug is admitted as Valid. This is the retained methodology; the test
# pins the behaviour so any future change to the rule order is visible.
assert classify_claim_quality("If metformin is contraindicated")[0] == "Valid"
print("Stage 12a quality filter PASS "
      "(note: drug-name override precedes the dangling-conditional rule)")

### Stage 12b — Heuristic verifier (similarity + lexical overlap + numeric consistency only)

In [ ]:
def verify_claim_against_evidence(claim_text: str, top_hit: dict) -> tuple[str, str]:
    """Heuristic evidence check for a claim that already passed the quality filter.

    Signals used: retrieval similarity, content-stem overlap, numeric consistency.
    NOT natural-language inference: no entailment, contradiction, or negation modelling.
    Returns (status, reason).
    """
    claim_nums = extract_numbers(claim_text)
    evidence_nums = extract_numbers(top_hit["text"])
    claim_stems = content_stems(claim_text)
    evidence_stems = content_stems(top_hit["text"])

    term_overlap = (len(claim_stems & evidence_stems) / len(claim_stems)) if claim_stems else 0.0
    similarity = top_hit["similarity"]
    chunk_id = top_hit["chunk_id"]

    numeric_conflict = bool(claim_nums) and bool(evidence_nums) and not (claim_nums & evidence_nums)

    if numeric_conflict:
        return "Unsupported", (
            f"Numeric mismatch: claim number(s) {sorted(claim_nums)} do not appear among "
            f"top-evidence number(s) {sorted(evidence_nums)} (chunk {chunk_id})."
        )

    if similarity < SIMILARITY_THRESHOLD or term_overlap < TERM_OVERLAP_THRESHOLD:
        return "Insufficient Evidence", (
            f"similarity={similarity:.3f} (threshold {SIMILARITY_THRESHOLD}), "
            f"term_overlap={term_overlap:.2f} (threshold {TERM_OVERLAP_THRESHOLD}); "
            f"evidence is related but does not adequately cover the claim's content."
        )

    return "Supported", (
        f"similarity={similarity:.3f}, term_overlap={term_overlap:.2f} "
        f"(majority of claim content matched, not a single incidental word), "
        f"no numeric conflict, against chunk {chunk_id}. "
        f"Lexical/numeric consistency only -- not proof of entailment."
    )

### Stage 12c — Run verification over every atomic claim

In [ ]:
claim_verification_records = []

for row in atomic_claims:
    # Unsupported/out-of-scope question placeholder -- no retrieval performed.
    if row["source_chunk_id"] is None:
        claim_verification_records.append({
            "question": row["question"],
            "original_extracted_span": None,
            "atomic_claim": row["atomic_claim"],
            "claim_quality": "Valid",
            "question_context_match": None,
            "retrieved_verification_chunk_id": None,
            "recommendation_number": None,
            "page": None,
            "source_url": None,
            "similarity_score": None,
            "verification_status": "Insufficient Evidence",
            "verification_reason": "Unsupported/out-of-scope question; no evidence retrieval performed.",
        })
        continue

    claim_text = row["atomic_claim"]
    quality, quality_reason = classify_claim_quality(claim_text)

    if quality == "Fragment":
        claim_verification_records.append({
            "question": row["question"],
            "original_extracted_span": row["original_extracted_span"],
            "atomic_claim": claim_text,
            "claim_quality": "Fragment",
            "question_context_match": None,
            "retrieved_verification_chunk_id": None,
            "recommendation_number": None,
            "page": None,
            "source_url": None,
            "similarity_score": None,
            "verification_status": "Insufficient Evidence",
            "verification_reason": quality_reason,
        })
        continue

    context_match, context_reason = check_question_context_match(row["question"], claim_text)
    hits = retrieve_top_k(claim_text, k=TOP_K)  # claim text is the query; no hard-coded IDs
    top_hit = hits[0]
    evidence_status, evidence_reason = verify_claim_against_evidence(claim_text, top_hit)

    if not context_match:
        final_status = "Insufficient Evidence"
        final_reason = f"Question-context mismatch: {context_reason}"
    else:
        final_status, final_reason = evidence_status, evidence_reason

    claim_verification_records.append({
        "question": row["question"],
        "original_extracted_span": row["original_extracted_span"],
        "atomic_claim": claim_text,
        "claim_quality": "Valid",
        "question_context_match": context_match,
        "retrieved_verification_chunk_id": top_hit["chunk_id"],
        "recommendation_number": top_hit["recommendation_number"],
        "page": top_hit["page"],
        "source_url": top_hit["source_url"],
        "similarity_score": round(top_hit["similarity"], 4),
        "verification_status": final_status,
        "verification_reason": final_reason,
    })

print(f"Stage 12: processed {len(claim_verification_records)} row(s).")

print("\n" + "=" * 100)
print("SAMPLE OF AUTOMATED CLAIM VERIFICATION (heuristic, NOT natural-language inference)")
print("=" * 100)
for row in [r for r in claim_verification_records
            if r["retrieved_verification_chunk_id"] is not None][:10]:
    print(f"\nQUESTION: {row['question']}")
    print(f"  atomic_claim:   {row['atomic_claim']}")
    print(f"  evidence chunk: {row['retrieved_verification_chunk_id']} "
          f"(rec {row['recommendation_number']}, p.{row['page']})")
    print(f"  similarity:     {row['similarity_score']}")
    print(f"  status:         {row['verification_status']}")
    print(f"  reason:         {row['verification_reason']}")

### Stage 12d — Validation, export, and final report

In [ ]:
from collections import Counter

quality_counts = Counter(r["claim_quality"] for r in claim_verification_records)
status_counts = Counter(r["verification_status"] for r in claim_verification_records)
context_mismatch_count = sum(
    1 for r in claim_verification_records if r["question_context_match"] is False
)

# Validation:
# 1. No Fragment row may carry any status other than Insufficient Evidence.
# 2. No row may be Supported unless claim_quality == Valid and context match is True.
verification_warnings = []
for r in claim_verification_records:
    if r["claim_quality"] == "Fragment" and r["verification_status"] != "Insufficient Evidence":
        verification_warnings.append(r)
    if r["verification_status"] == "Supported" and not (
        r["claim_quality"] == "Valid" and r["question_context_match"] is True
    ):
        verification_warnings.append(r)

if not atomic_claims:
    stage12_status = "SKIPPED"  # nothing to verify (Stage 10/11 did not produce claims)
else:
    stage12_status = (
        "PASS"
        if len(claim_verification_records) == len(atomic_claims) and not verification_warnings
        else "FAIL"
    )

print("=" * 60)
print("STAGE 12 SUMMARY")
print("=" * 60)
print(f"Valid claims:            {quality_counts.get('Valid', 0)}")
print(f"Fragment claims:         {quality_counts.get('Fragment', 0)}")
print("-" * 60)
print(f"Supported:               {status_counts.get('Supported', 0)}")
print(f"Unsupported:             {status_counts.get('Unsupported', 0)}")
print(f"Insufficient Evidence:   {status_counts.get('Insufficient Evidence', 0)}")
print("-" * 60)
print(f"Context mismatches:      {context_mismatch_count}")
print(f"Validation warnings:     {len(verification_warnings)}")
print(f"Stage 12 status:         {stage12_status}")
print("=" * 60)

# --- Export (automated set only - never overwrites the manual gold files) --
automated_claim_df = pd.DataFrame(claim_verification_records)
automated_claim_df.to_csv(AUTOMATED_CLAIMS_CSV, index=False)
with open(AUTOMATED_CLAIMS_JSON, "w") as f:
    json.dump(claim_verification_records, f, indent=2)

print(f"\nSaved {len(automated_claim_df)} AUTOMATED rows to "
      f"{AUTOMATED_CLAIMS_CSV} and {AUTOMATED_CLAIMS_JSON}")
assert AUTOMATED_CLAIMS_CSV != MANUAL_CLAIMS_CSV, "Automated output must not overwrite manual gold."
automated_claim_df.head()

In [ ]:
LIMITATIONS = [
    "Stage 12 is a heuristic verifier. It uses only retrieval similarity, content-stem "
    "lexical overlap, and numerical consistency. It does not perform natural-language "
    "inference and has no model of entailment, contradiction, or negation.",
    "A 'Supported' label means the retrieved guideline text is lexically and numerically "
    "consistent with the claim. It is not evidence of clinical correctness or safety.",
    "'Unsupported' is assigned only on an explicit numeric conflict, so the class is "
    "under-detected by construction.",
    "The corpus is an 18-chunk excerpt of NICE NG28, not the full guideline. "
    "The 5 unsupported questions are out of scope for this excerpt by design.",
    "Stage 10 produces short extractive spans copied from retrieved text, not "
    "clinically complete answers.",
    "Hit@3 is measured on 15 answerable questions; the sample is small and the "
    "gold chunk for each question was assigned by a single annotator.",
    "The manual gold records and the automated pipeline output are reported separately "
    "and are not treated as interchangeable.",
]

final_summary = {
    "project_title": (
        "Multi-Stage Heuristic and NLI Ensemble for Claim-Level Hallucination Control "
        "in Retrieval-Augmented LLM Medical QA: A Reproducible Pilot Study (NICE NG28)"
    ),
    "corpus": {
        "document_id": corpus_records[0]["document_id"],
        "n_chunks": len(corpus_records),
        "sections": corpus_validation_report["sections"],
        "text_sha256": corpus_validation_report["corpus_text_sha256"],
    },
    "retrieval": {
        "embedding_model": EMBEDDING_MODEL_NAME,
        "top_k": TOP_K,
        "smoke_test_score": smoke_test_score,
        "n_answerable": len(answerable_results),
        "n_unsupported": len(unsupported_results),
        "hit_at_3": hit_at_3_score,
    },
    "extractive_qa": {"model": QA_MODEL_NAME, "status": stage10_status},
    "claim_decomposition": {
        "method": "rule-based, no LLM",
        "n_atomic_claims": len(answerable_claim_rows),
        "status": stage11_status,
    },
    "claim_verification": {
        "method": "heuristic: similarity + lexical overlap + numeric consistency ONLY "
                  "(not natural-language inference)",
        "similarity_threshold": SIMILARITY_THRESHOLD,
        "term_overlap_threshold": TERM_OVERLAP_THRESHOLD,
        "quality_counts": dict(quality_counts),
        "status_counts": dict(status_counts),
        "status": stage12_status,
    },
    "manual_reference_set": {
        "n_records": len(manual_claim_records),
        "provenance": "human annotator; never produced or modified by a model",
        "validation_warnings": len(manual_validation_warnings),
    },
    "limitations": LIMITATIONS,
}

with open(FINAL_SUMMARY_JSON, "w") as f:
    json.dump(final_summary, f, indent=2)

report_lines = [
    "# Final Report",
    "",
    f"**{final_summary['project_title']}**",
    "",
    "## 1. Pipeline status",
    "",
    "| Stage | Status |",
    "|---|---|",
    f"| 3 Corpus validation | {corpus_validation_report['status']} |",
    f"| 8 Retrieval evaluation | Hit@{TOP_K} = {hit_at_3_score:.2%} |",
    f"| 10 Extractive QA | {stage10_status} |",
    f"| 11 Claim decomposition | {stage11_status} |",
    f"| 12 Claim verification | {stage12_status} |",
    "",
    "## 2. Retrieval",
    "",
    f"- Corpus: {len(corpus_records)} verified NICE NG28 chunks across "
    f"{len(corpus_validation_report['sections'])} sections.",
    f"- Smoke test (5 questions, sanity check only): {smoke_test_score:.0%}.",
    f"- Main evaluation set: {len(EVAL_QUESTIONS)} questions "
    f"({len(answerable_results)} answerable, {len(unsupported_results)} out-of-scope).",
    f"- **Hit@{TOP_K} on answerable questions: {hits_count}/{len(answerable_results)} "
    f"= {hit_at_3_score:.2%}.**",
    "- Out-of-scope questions are retained and reported separately; they carry no gold "
    "chunk and are excluded from Hit@3.",
    "",
    "## 3. Claim-level verification (automated)",
    "",
    f"- Atomic claims verified: {len(answerable_claim_rows)}.",
    f"- Supported: {status_counts.get('Supported', 0)} | "
    f"Unsupported: {status_counts.get('Unsupported', 0)} | "
    f"Insufficient Evidence: {status_counts.get('Insufficient Evidence', 0)}.",
    f"- Claims rejected by the quality filter as fragments: {quality_counts.get('Fragment', 0)}.",
    "",
    "## 4. Manual reference set",
    "",
    f"- {len(manual_claim_records)} human-written gold/reference claim records, "
    f"{len(manual_validation_warnings)} validation warning(s).",
    "- Reported separately from the automated output; the two are never merged.",
    "",
    "## 5. Limitations",
    "",
]
report_lines += [f"- {limitation}" for limitation in LIMITATIONS]

with open(FINAL_REPORT_MD, "w") as f:
    f.write("\n".join(report_lines) + "\n")

print(f"Wrote {FINAL_REPORT_MD} and {FINAL_SUMMARY_JSON}")
print()
print("\n".join(report_lines))

---
# Stage 13 — Stage 4 V2: Reference-Claim Verification (recovered from archive)

**Input:** `manual_claim_records` (Stage 9b, human-written gold claims).
**Output:** `REFERENCE_CLAIM_ROWS`, per-status Precision/Recall/F1,
`stage4v2_reference_claim_comparison.csv` / `.json`.

This is the pre-refactor **"Stage 4 V2"** stage, recovered from
`ng28_archive_removed_code.ipynb` (Group 2, section 2.1). Stage 12 verifies
**machine-extracted** atomic claims (from Stage 11's rule-based decomposition). This
stage instead runs the **same automated verifier functions from Stage 12**
(`classify_claim_quality`, `check_question_context_match`,
`verify_claim_against_evidence`) directly against the **human-written** claim text
from Stage 9b, so the automated verifier can be compared claim-for-claim against a
human gold label.

This stage is read-only with respect to Stage 12: it reuses Stage 12's verifier
functions but does not modify `claim_verification_records` or any Stage 12 output
file. Output files are saved under new `stage4v2_*` names so nothing from Stage 12
is overwritten.


In [ ]:
# --- Stage 13a: Build REFERENCE_CLAIM_ROWS (Stage 4 V2 core) ---------------
# Verifies the HUMAN-WRITTEN claim text from Stage 9b (manual_claim_records)
# directly, using the SAME verifier functions Stage 12 uses on machine-extracted
# claims: classify_claim_quality, check_question_context_match,
# verify_claim_against_evidence. No new verifier logic is introduced here.

REFERENCE_CLAIM_ROWS = []

for rec in manual_claim_records:
    question = rec["question"]
    claim_text = rec["atomic_claim"]
    human_status = rec["status"]
    human_chunk_id = rec["supporting_chunk_id"]

    if human_chunk_id is None:
        # One of the 5 unsupported-question placeholders -- not a real
        # checkable claim. No retrieval is performed; kept for reporting
        # only, excluded from the metric calculation below.
        REFERENCE_CLAIM_ROWS.append({
            "record_type": "unsupported_placeholder",
            "question": question,
            "atomic_claim": claim_text,
            "human_status": human_status,
            "automated_status": "Insufficient Evidence",
            "human_supporting_chunk_id": None,
            "automated_retrieved_chunk_id": None,
            "recommendation_number": None,
            "page": None,
            "similarity_score": None,
            "claim_quality": "Valid",
            "question_context_match": None,
            "agreement": human_status == "Insufficient Evidence",
            "verifier_reason": "Unsupported/out-of-scope question placeholder; no evidence retrieval performed.",
            "excluded_from_metrics": True,
        })
        continue

    quality, quality_reason = classify_claim_quality(claim_text)

    if quality == "Fragment":
        automated_status = "Insufficient Evidence"
        reason = quality_reason
        top_hit = None
        context_match = None
    else:
        context_match, context_reason = check_question_context_match(question, claim_text)
        hits = retrieve_top_k(claim_text, k=TOP_K)  # reference claim text used directly -- not a Stage 11 span
        top_hit = hits[0]
        evidence_status, evidence_reason = verify_claim_against_evidence(claim_text, top_hit)
        if not context_match:
            automated_status = "Insufficient Evidence"
            reason = f"Question-context mismatch: {context_reason}"
        else:
            automated_status = evidence_status
            reason = evidence_reason

    REFERENCE_CLAIM_ROWS.append({
        "record_type": "human_reference",
        "question": question,
        "atomic_claim": claim_text,
        "human_status": human_status,
        "automated_status": automated_status,
        "human_supporting_chunk_id": human_chunk_id,
        "automated_retrieved_chunk_id": top_hit["chunk_id"] if top_hit else None,
        "recommendation_number": top_hit["recommendation_number"] if top_hit else None,
        "page": top_hit["page"] if top_hit else None,
        "similarity_score": round(top_hit["similarity"], 4) if top_hit else None,
        "claim_quality": quality,
        "question_context_match": context_match,
        "agreement": human_status == automated_status,
        "verifier_reason": reason,
        "excluded_from_metrics": False,
    })

n_placeholders = sum(1 for r in REFERENCE_CLAIM_ROWS if r["record_type"] == "unsupported_placeholder")
n_reference = sum(1 for r in REFERENCE_CLAIM_ROWS if r["record_type"] == "human_reference")
assert len(REFERENCE_CLAIM_ROWS) == len(manual_claim_records)
print(f"Stage 13 (Stage 4 V2): verified {n_reference} human reference claim(s) directly "
      f"(plus {n_placeholders} unsupported-question placeholder(s), reported but excluded from metrics).")


In [ ]:
# --- Stage 13b: Detailed listing -------------------------------------------
print("=" * 100)
print("STAGE 13 / STAGE 4 V2 -- REFERENCE-CLAIM VERIFICATION (human claim text verified directly)")
print("=" * 100)

for row in REFERENCE_CLAIM_ROWS:
    print(f"\n[{row['record_type']}] QUESTION: {row['question']}")
    print(f"  atomic_claim:                 {row['atomic_claim']}")
    print(f"  human_status:                 {row['human_status']}")
    print(f"  automated_status:             {row['automated_status']}")
    print(f"  agreement:                    {row['agreement']}")
    print(f"  human_supporting_chunk_id:    {row['human_supporting_chunk_id']}")
    print(f"  automated_retrieved_chunk_id: {row['automated_retrieved_chunk_id']}")
    print(f"  similarity_score:             {row['similarity_score']}")
    print(f"  verifier_reason:              {row['verifier_reason']}")
print("\n" + "=" * 100)


In [ ]:
# --- Stage 13c: Per-status Precision / Recall / F1 (24 human reference claims) ---
ALL_STATUSES_LIST = ["Supported", "Unsupported", "Insufficient Evidence"]

metric_rows = [r for r in REFERENCE_CLAIM_ROWS if not r["excluded_from_metrics"]]
n_metric_rows = len(metric_rows)

per_status_results = {}
f1_scores_for_macro = []

print("=" * 60)
print("PER-STATUS PRECISION / RECALL / F1 (24 human reference claims)")
print("=" * 60)

for status in ALL_STATUSES_LIST:
    n_gold = sum(1 for r in metric_rows if r["human_status"] == status)

    if n_gold == 0:
        print(f"\n{status}:")
        print("  Not evaluable: no gold examples for this status.")
        per_status_results[status] = None
        continue

    tp = sum(1 for r in metric_rows if r["human_status"] == status and r["automated_status"] == status)
    fp = sum(1 for r in metric_rows if r["human_status"] != status and r["automated_status"] == status)
    fn = sum(1 for r in metric_rows if r["human_status"] == status and r["automated_status"] != status)

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) > 0 else 0.0

    per_status_results[status] = {"precision": precision, "recall": recall, "f1": f1, "n_gold": n_gold}
    f1_scores_for_macro.append(f1)

    print(f"\n{status}: (n_gold={n_gold})")
    print(f"  Precision: {precision:.3f}")
    print(f"  Recall:    {recall:.3f}")
    print(f"  F1:        {f1:.3f}")

macro_f1 = sum(f1_scores_for_macro) / len(f1_scores_for_macro) if f1_scores_for_macro else None

print("\n" + "-" * 60)
if macro_f1 is not None:
    print(f"Macro-F1 (averaged over {len(f1_scores_for_macro)} evaluable status(es)): {macro_f1:.3f}")
else:
    print("Macro-F1 not reported: no status had gold examples.")
print("=" * 60)


In [ ]:
# --- Stage 13d: Export (new file names -- does not touch any Stage 1-12 output) ---
# STAGE4V2_REFERENCE_CSV / STAGE4V2_REFERENCE_JSON now come from Stage 1.
stage4v2_reference_df = pd.DataFrame(REFERENCE_CLAIM_ROWS)
stage4v2_reference_df.to_csv(STAGE4V2_REFERENCE_CSV, index=False)
with open(STAGE4V2_REFERENCE_JSON, "w") as f:
    json.dump(REFERENCE_CLAIM_ROWS, f, indent=2)

for existing in (MANUAL_CLAIMS_CSV, MANUAL_CLAIMS_JSON, AUTOMATED_CLAIMS_CSV, AUTOMATED_CLAIMS_JSON):
    assert STAGE4V2_REFERENCE_CSV != existing and STAGE4V2_REFERENCE_JSON != existing, (
        "Stage 13 output must not overwrite any Stage 1-12 output file."
    )

agreement_count = sum(1 for r in metric_rows if r["agreement"])
agreement_pct = (agreement_count / n_metric_rows * 100) if n_metric_rows else 0.0
stage13_status = "PASS" if len(REFERENCE_CLAIM_ROWS) == len(manual_claim_records) else "FAIL"

print(f"Saved {len(stage4v2_reference_df)} rows to {STAGE4V2_REFERENCE_CSV} and {STAGE4V2_REFERENCE_JSON}")
print("=" * 60)
print("STAGE 13 / STAGE 4 V2 SUMMARY")
print("=" * 60)
print(f"Total reference claims (metric set):              {n_metric_rows}")
print(f"Automated claims verified (incl. 5 placeholders):  {len(REFERENCE_CLAIM_ROWS)}")
print(f"Agreement count (metric set):                      {agreement_count}/{n_metric_rows}")
print(f"Agreement percentage:                              {agreement_pct:.1f}%")
print("-" * 60)
print("Per-status precision/recall/F1:")
for status in ALL_STATUSES_LIST:
    result = per_status_results[status]
    if result is None:
        print(f"  {status}: Not evaluable: no gold examples for this status.")
    else:
        p = result["precision"]; r = result["recall"]; f1v = result["f1"]; ng = result["n_gold"]
        print(f"  {status}: P={p:.3f} R={r:.3f} F1={f1v:.3f} (n_gold={ng})")
print(f"  Macro-F1: {'not reported' if macro_f1 is None else f'{macro_f1:.3f}'}")
print("-" * 60)
print(f"Stage 13 status: {stage13_status}")
print("=" * 60)
print("LIMITATION: The reference claims are single-annotator and the verifier is a")
print("lightweight heuristic (Stage 12's), not a true NLI model. See Stage 15 for NLI.")
print("=" * 60)

stage4v2_reference_df


---
# Stage 14 — Adversarial Claims Evaluation (7 deliberately contradictory claims, recovered from archive)

**Input:** none (self-contained claim set), Stage 12's verifier functions.
**Output:** `ADVERSARIAL_CLAIMS`, `ADVERSARIAL_RESULTS`,
`stage4v2_adversarial_claims.csv` / `.json`.

The 24 genuine human reference claims (Stage 13) are all gold-`Supported`, so without
a deliberately-wrong claim set there are **zero** gold `Unsupported` examples anywhere
in this notebook. This stage adds 7 adversarial claims -- each contradicting a
specific number or treatment named in the NG28 corpus -- and checks whether Stage 12's
heuristic verifier flags them as `Unsupported`. 5 conflicts are numeric (a threshold,
dose, or frequency is changed); 2 are non-numeric drug/treatment substitutions, which
the heuristic verifier is not designed to catch (it only detects numeric mismatches).


In [ ]:
# --- Stage 14a: Define the 7 adversarial claims and evaluate them ----------
ADVERSARIAL_CLAIMS = [
    {
        "question": "What is the HbA1c target for type 2 diabetes?",
        "adversarial_claim": "Adults managed by diet alone should aim for an HbA1c of 42 mmol/mol (6.0%).",
        "true_fact": "The guideline states an HbA1c target of 48 mmol/mol (6.5%) for adults managed by "
                     "diet alone (NICE_NG28_006).",
        "conflict_type": "numeric",
    },
    {
        "question": "When should treatment be intensified?",
        "adversarial_claim": "Treatment should be intensified if HbA1c rises to 65 mmol/mol (8.0%) or higher.",
        "true_fact": "The guideline states the intensification threshold is 58 mmol/mol (7.5%) or higher "
                     "(NICE_NG28_007).",
        "conflict_type": "numeric",
    },
    {
        "question": "How often should HbA1c levels be measured?",
        "adversarial_claim": "HbA1c should be measured every 12 months once stable on unchanging therapy.",
        "true_fact": "The guideline states HbA1c should be measured every 6 months once stable on "
                     "unchanging therapy (NICE_NG28_005).",
        "conflict_type": "numeric",
    },
    {
        "question": "What treatment should be offered to adults with atherosclerotic cardiovascular disease?",
        "adversarial_claim": "Adults with atherosclerotic cardiovascular disease should be offered "
                              "subcutaneous semaglutide up to 5 mg once a week.",
        "true_fact": "The guideline states subcutaneous semaglutide up to 1 mg once a week "
                     "(NICE_NG28_011).",
        "conflict_type": "numeric",
    },
    {
        "question": "Under what criteria should isCGM be offered to adults on multiple daily insulin injections?",
        "adversarial_claim": "isCGM should be offered to adults who self-measure at least 3 times a day.",
        "true_fact": "The guideline states 'at least 8 times a day' as one qualifying criterion "
                     "(NICE_NG28_018).",
        "conflict_type": "numeric",
    },
    {
        "question": "What first-line medicines should be offered if there is no relevant comorbidity?",
        "adversarial_claim": "Adults with no relevant comorbidity should be offered a sulfonylurea and "
                              "pioglitazone as first-line treatment.",
        "true_fact": "The guideline states modified-release metformin and an SGLT-2 inhibitor "
                     "(NICE_NG28_008) -- a different treatment, not a numeric value.",
        "conflict_type": "non-numeric (drug substitution)",
    },
    {
        "question": "What should be offered if metformin is contraindicated or not tolerated?",
        "adversarial_claim": "If metformin is contraindicated or not tolerated, adults should be offered "
                              "insulin twice daily.",
        "true_fact": "The guideline states monotherapy with an SGLT-2 inhibitor (NICE_NG28_009) -- a "
                     "different treatment, not a numeric value.",
        "conflict_type": "non-numeric (drug substitution)",
    },
]

ADVERSARIAL_RESULTS = []

for item in ADVERSARIAL_CLAIMS:
    claim_text = item["adversarial_claim"]
    question = item["question"]

    quality, quality_reason = classify_claim_quality(claim_text)
    if quality == "Fragment":
        automated_status, reason, top_hit, context_match = "Insufficient Evidence", quality_reason, None, None
    else:
        context_match, context_reason = check_question_context_match(question, claim_text)
        hits = retrieve_top_k(claim_text, k=TOP_K)
        top_hit = hits[0]
        evidence_status, evidence_reason = verify_claim_against_evidence(claim_text, top_hit)
        if not context_match:
            automated_status, reason = "Insufficient Evidence", f"Question-context mismatch: {context_reason}"
        else:
            automated_status, reason = evidence_status, evidence_reason

    detected = automated_status == "Unsupported"
    ADVERSARIAL_RESULTS.append({
        "record_type": "adversarial",
        "question": question,
        "atomic_claim": claim_text,
        "true_fact": item["true_fact"],
        "conflict_type": item["conflict_type"],
        "human_status": "Unsupported",  # expected, by design of the adversarial claim
        "automated_status": automated_status,
        "human_supporting_chunk_id": None,
        "automated_retrieved_chunk_id": top_hit["chunk_id"] if top_hit else None,
        "recommendation_number": top_hit["recommendation_number"] if top_hit else None,
        "page": top_hit["page"] if top_hit else None,
        "similarity_score": round(top_hit["similarity"], 4) if top_hit else None,
        "claim_quality": quality,
        "question_context_match": context_match,
        "agreement": detected,
        "verifier_reason": reason,
        "excluded_from_metrics": True,
    })

print("=" * 100)
print("STAGE 14 -- ADVERSARIAL EVALUATION (deliberately contradictory claims)")
print("=" * 100)
for r in ADVERSARIAL_RESULTS:
    print(f"\nQUESTION: {r['question']}")
    print(f"  adversarial_claim: {r['atomic_claim']}")
    print(f"  true_fact:         {r['true_fact']}")
    print(f"  conflict_type:     {r['conflict_type']}")
    print(f"  automated_status:  {r['automated_status']}  "
          f"(\'Unsupported\' expected -> detected: {r['agreement']})")
    print(f"  verifier_reason:   {r['verifier_reason']}")
print("\n" + "=" * 100)

n_detected = sum(1 for r in ADVERSARIAL_RESULTS if r["agreement"])
print(f"Adversarial claims correctly flagged as Unsupported: {n_detected}/{len(ADVERSARIAL_RESULTS)}")
print("Non-numeric contradictions the heuristic misses (by design limitation) are expected here --")
print("this heuristic can only catch numeric mismatches, not drug/treatment-name contradictions.")


In [ ]:
# --- Stage 14b: Export (new file names -- does not touch Stage 13 or earlier output) ---
# STAGE4V2_ADVERSARIAL_CSV / STAGE4V2_ADVERSARIAL_JSON now come from Stage 1.
stage4v2_adversarial_df = pd.DataFrame(ADVERSARIAL_RESULTS)
stage4v2_adversarial_df.to_csv(STAGE4V2_ADVERSARIAL_CSV, index=False)
with open(STAGE4V2_ADVERSARIAL_JSON, "w") as f:
    json.dump(ADVERSARIAL_RESULTS, f, indent=2)

for existing in (STAGE4V2_REFERENCE_CSV, STAGE4V2_REFERENCE_JSON,
                  MANUAL_CLAIMS_CSV, MANUAL_CLAIMS_JSON,
                  AUTOMATED_CLAIMS_CSV, AUTOMATED_CLAIMS_JSON):
    assert STAGE4V2_ADVERSARIAL_CSV != existing and STAGE4V2_ADVERSARIAL_JSON != existing, (
        "Stage 14 output must not overwrite any earlier output file."
    )

n_numeric = sum(1 for r in ADVERSARIAL_RESULTS if r["conflict_type"] == "numeric")
n_nonnumeric = len(ADVERSARIAL_RESULTS) - n_numeric
stage14_status = "PASS" if len(ADVERSARIAL_RESULTS) == len(ADVERSARIAL_CLAIMS) == 7 else "FAIL"

print(f"Saved {len(stage4v2_adversarial_df)} rows to {STAGE4V2_ADVERSARIAL_CSV} and {STAGE4V2_ADVERSARIAL_JSON}")
print("=" * 60)
print("STAGE 14 SUMMARY")
print("=" * 60)
print(f"Adversarial claims:                 {len(ADVERSARIAL_CLAIMS)} (5 numeric expected, 2 non-numeric expected)")
print(f"  numeric conflicts:                {n_numeric}")
print(f"  non-numeric (drug) conflicts:      {n_nonnumeric}")
print(f"Correctly flagged Unsupported:      {n_detected}/{len(ADVERSARIAL_RESULTS)}")
for r in ADVERSARIAL_RESULTS:
    ctype = r["conflict_type"]; agr = r["agreement"]; ctext = r["atomic_claim"][:70]
    print(f"  [{ctype:<28}] detected={agr}: {ctext}")
print(f"Stage 14 status: {stage14_status}")
print("=" * 60)
print("LIMITATION: only 7 adversarial claims -- too small a sample to generalise an")
print("accuracy estimate beyond this test set. See Stage 15 for the NLI comparison.")
print("=" * 60)

stage4v2_adversarial_df


---
# Stage 15 — Stage 4 V3: NLI-Based Claim Verification (recovered from archive)

**Input:** `manual_claim_records` (Stage 9b), `REFERENCE_CLAIM_ROWS` (Stage 13),
`ADVERSARIAL_CLAIMS` / `ADVERSARIAL_RESULTS` (Stage 14).
**Output:** `STAGE4_V3_RECORDS`, `stage4v3_nli_comparison.csv` / `.json`,
`stage4v3_nli_summary.json`.

Stage 12's heuristic verifier and Stage 13's Stage-4-V2 verifier both check only
retrieval similarity, lexical overlap, and numeric consistency -- **not** entailment,
contradiction, or negation. This stage is the **only** component in the notebook that
performs true natural-language inference: a local, free, offline
`cross-encoder/nli-MiniLM2-L6-H768` model, loaded directly with `AutoTokenizer` +
`AutoModelForSequenceClassification` (no `pipeline()`, no API key). It is deliberately
kept separate from Stage 12 so Stage 12's documented "heuristic only" limitation
stays accurate. It evaluates all 36 claims from Stages 13-14 (29 human gold + 7
adversarial) so the NLI verifier's accuracy can be compared directly against the
heuristic verifier's accuracy on the same claims.


In [ ]:
# --- Stage 15a: Load the local NLI model ------------------------------------
!pip install -q "transformers[torch]" --upgrade

import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# STAGE4_V3_MODEL_NAME now comes from Stage 1.

STAGE4_V3_MODEL_LOADED = False
STAGE4_V3_LOAD_ERROR = None
stage4_v3_tokenizer = None
stage4_v3_model = None
STAGE4_V3_ID2LABEL = None

try:
    stage4_v3_tokenizer = AutoTokenizer.from_pretrained(STAGE4_V3_MODEL_NAME)
    stage4_v3_model = AutoModelForSequenceClassification.from_pretrained(STAGE4_V3_MODEL_NAME)
    stage4_v3_model.eval()

    raw_id2label = stage4_v3_model.config.id2label
    STAGE4_V3_ID2LABEL = {int(k): str(v).strip().lower() for k, v in raw_id2label.items()}

    labels_found = set(STAGE4_V3_ID2LABEL.values())
    expected_labels = {"contradiction", "entailment", "neutral"}
    if not expected_labels.issubset(labels_found):
        raise ValueError(
            f"Unexpected label set from model config: {labels_found}. "
            f"Expected to find: {expected_labels}."
        )

    STAGE4_V3_MODEL_LOADED = True
except Exception as exc:
    STAGE4_V3_LOAD_ERROR = f"{type(exc).__name__}: {exc}"

print("=" * 60)
print("STAGE 15 / STAGE 4 V3 -- NLI MODEL LOAD CHECK")
print("=" * 60)
print(f"Model:      {STAGE4_V3_MODEL_NAME}")
print(f"Loaded via: AutoTokenizer + AutoModelForSequenceClassification (no pipeline, no API)")
if STAGE4_V3_MODEL_LOADED:
    print(f"id2label:   {STAGE4_V3_ID2LABEL}")
else:
    print(f"Load error: {STAGE4_V3_LOAD_ERROR}")
print(f"Status: {'PASS' if STAGE4_V3_MODEL_LOADED else 'FAIL'}")
print("=" * 60)


In [ ]:
# --- Stage 15b: NLI forward pass + inference smoke test ---------------------
def stage4_v3_nli_probs(premise: str, hypothesis: str) -> dict:
    """
    Single local NLI forward pass: premise = retrieved guideline text,
    hypothesis = the claim. Returns softmax scores per relation label.
    These are NLI relation scores, not truth probabilities.
    """
    inputs = stage4_v3_tokenizer(premise, hypothesis, return_tensors="pt", truncation=True)
    with torch.no_grad():
        logits = stage4_v3_model(**inputs).logits[0]
    probs = torch.softmax(logits, dim=-1).tolist()
    return {STAGE4_V3_ID2LABEL[i]: probs[i] for i in range(len(probs))}


STAGE4_V3_SMOKE_TESTS = [
    {
        "premise": "Measure HbA1c levels in adults with type 2 diabetes every 6 months "
                   "once the HbA1c level and blood glucose lowering therapy are stable.",
        "hypothesis": "HbA1c should be measured every 6 months once stable.",
        "expected_top_label": "entailment",
    },
    {
        "premise": "For adults whose type 2 diabetes is managed by healthy living and diet, "
                   "support them to aim for an HbA1c level of 48 mmol/mol (6.5%).",
        "hypothesis": "Adults managed by diet alone should aim for an HbA1c of 42 mmol/mol (6.0%).",
        "expected_top_label": "contradiction",
    },
    {
        "premise": "Advise adults with type 2 diabetes at their annual review that they are "
                   "at higher risk of periodontitis.",
        "hypothesis": "Adults with type 2 diabetes should be referred to a dietitian within "
                       "one week of diagnosis.",
        "expected_top_label": "neutral",
    },
]

stage4_v3_inference_status = "PASS"

print("=" * 60)
print("STAGE 15 / STAGE 4 V3 -- NLI INFERENCE SMOKE TEST")
print("=" * 60)

if not STAGE4_V3_MODEL_LOADED:
    stage4_v3_inference_status = "FAIL"
    print("Skipped: model did not load (see the load check above).")
else:
    for case in STAGE4_V3_SMOKE_TESTS:
        probs = stage4_v3_nli_probs(case["premise"], case["hypothesis"])
        rounded_probs = {k: round(v, 3) for k, v in probs.items()}
        predicted_label = max(probs, key=probs.get)
        ok = predicted_label == case["expected_top_label"]
        if not ok:
            stage4_v3_inference_status = "FAIL"
        print(f"\npremise:    {case['premise']}")
        print(f"hypothesis: {case['hypothesis']}")
        print(f"probs:      {rounded_probs}")
        print(f"expected:   {case['expected_top_label']}  |  predicted: {predicted_label}  "
              f"|  {'OK' if ok else 'MISMATCH'}")

print("\n" + "-" * 60)
print(f"Stage 15 NLI inference smoke test status: {stage4_v3_inference_status}")
print("=" * 60)


In [ ]:
# --- Stage 15c: Claim-level NLI classifier (top-3 retrieved chunks, best relation) ---
def stage4_v3_classify_claim(claim_text: str, hits: list[dict]) -> dict:
    """
    Local NLI (premise = each retrieved chunk's text, hypothesis = claim_text) over
    the top-3 retrieved chunks independently; keeps the chunk with the strongest
    entailment-or-contradiction score. Scores are NLI relation scores from a
    general-language model, not truth probabilities.
    """
    best = None
    for hit in hits:
        probs = stage4_v3_nli_probs(hit["text"], claim_text)
        strongest_label = max(probs, key=probs.get)
        relevant_score = max(probs["entailment"], probs["contradiction"])
        candidate = {
            "chunk_id": hit["chunk_id"],
            "recommendation_number": hit["recommendation_number"],
            "page": hit["page"],
            "source_url": hit["source_url"],
            "entailment": probs["entailment"],
            "contradiction": probs["contradiction"],
            "neutral": probs["neutral"],
            "strongest_label": strongest_label,
            "relevant_score": relevant_score,
        }
        if best is None or candidate["relevant_score"] > best["relevant_score"]:
            best = candidate

    entail = best["entailment"]
    contra = best["contradiction"]
    neutral = best["neutral"]

    if entail >= NLI_ENTAILMENT_THRESHOLD and best["strongest_label"] == "entailment":
        status = "Supported"
        reason = (f"Chunk {best['chunk_id']}: entailment={entail:.3f} >= {NLI_ENTAILMENT_THRESHOLD} and is the "
                   f"strongest relation (contradiction={contra:.3f}, neutral={neutral:.3f}).")
    elif contra >= NLI_CONTRADICTION_THRESHOLD and best["strongest_label"] == "contradiction":
        status = "Unsupported"
        reason = (f"Chunk {best['chunk_id']}: contradiction={contra:.3f} >= {NLI_CONTRADICTION_THRESHOLD} and is the "
                   f"strongest relation (entailment={entail:.3f}, neutral={neutral:.3f}).")
    else:
        status = "Insufficient Evidence"
        reason = (f"Chunk {best['chunk_id']}: strongest relation is '{best['strongest_label']}' "
                   f"(entailment={entail:.3f}, contradiction={contra:.3f}, neutral={neutral:.3f}) "
                   f"-- neither threshold condition was met.")

    return {
        "nli_status": status,
        "entailment_score": round(entail, 4),
        "contradiction_score": round(contra, 4),
        "neutral_score": round(neutral, 4),
        "selected_chunk_id": best["chunk_id"],
        "recommendation_number": best["recommendation_number"],
        "page": best["page"],
        "source_url": best["source_url"],
        "nli_reason": reason,
    }


In [ ]:
# --- Stage 15d: Build the 36-claim evaluation queue (29 human gold + 7 adversarial) ---
STAGE4_V3_EVAL_ITEMS = []

assert len(manual_claim_records) == len(REFERENCE_CLAIM_ROWS) == 29, (
    "Expected 29 human gold/reference records -- do not proceed if this changed."
)
assert len(ADVERSARIAL_CLAIMS) == len(ADVERSARIAL_RESULTS) == 7, (
    "Expected 7 adversarial claims -- do not proceed if this changed."
)

for gold_rec, heur_row in zip(manual_claim_records, REFERENCE_CLAIM_ROWS):
    assert gold_rec["question"] == heur_row["question"]
    assert gold_rec["atomic_claim"] == heur_row["atomic_claim"]
    STAGE4_V3_EVAL_ITEMS.append({
        "question": gold_rec["question"],
        "atomic_claim": gold_rec["atomic_claim"],
        "claim_type": "human_gold",
        "human_gold_status": gold_rec["status"],
        "heuristic_status": heur_row["automated_status"],
    })

for adv_item, adv_result in zip(ADVERSARIAL_CLAIMS, ADVERSARIAL_RESULTS):
    assert adv_item["question"] == adv_result["question"]
    STAGE4_V3_EVAL_ITEMS.append({
        "question": adv_item["question"],
        "atomic_claim": adv_item["adversarial_claim"],
        "claim_type": "adversarial",
        "human_gold_status": "Unsupported",  # expected by design of the adversarial set
        "heuristic_status": adv_result["automated_status"],
    })

print(f"Stage 15 evaluation queue: {len(STAGE4_V3_EVAL_ITEMS)} claims (expected 29 + 7 = 36)")
print(f"  human_gold:  {sum(1 for i in STAGE4_V3_EVAL_ITEMS if i['claim_type'] == 'human_gold')}")
print(f"  adversarial: {sum(1 for i in STAGE4_V3_EVAL_ITEMS if i['claim_type'] == 'adversarial')}")


In [ ]:
# --- Stage 15e: Run NLI verification over the 36-claim queue ----------------
STAGE4_V3_RECORDS = []

stage4_v3_prereqs_ok = STAGE4_V3_MODEL_LOADED and (stage4_v3_inference_status == "PASS")

if not stage4_v3_prereqs_ok:
    print("Skipping Stage 15 evaluation: model load or inference smoke test did not PASS.")
else:
    for item in STAGE4_V3_EVAL_ITEMS:
        hits = retrieve_top_k(item["atomic_claim"], k=3)  # unchanged retriever
        nli_result = stage4_v3_classify_claim(item["atomic_claim"], hits)
        STAGE4_V3_RECORDS.append({
            "question": item["question"],
            "atomic_claim": item["atomic_claim"],
            "claim_type": item["claim_type"],
            "human_gold_status": item["human_gold_status"],
            "heuristic_status": item["heuristic_status"],
            "nli_status": nli_result["nli_status"],
            "contradiction_score": nli_result["contradiction_score"],
            "entailment_score": nli_result["entailment_score"],
            "neutral_score": nli_result["neutral_score"],
            "selected_chunk_id": nli_result["selected_chunk_id"],
            "recommendation_number": nli_result["recommendation_number"],
            "page": nli_result["page"],
            "source_url": nli_result["source_url"],
            "nli_reason": nli_result["nli_reason"],
        })

    print(f"Stage 15 evaluation complete: {len(STAGE4_V3_RECORDS)} claim(s) evaluated.")


In [ ]:
# --- Stage 15f: Metrics -- heuristic vs NLI on the same 36 claims -----------
stage4_v3_supported_agreement = None
stage4_v3_insufficient_agreement = None
stage4_v3_adversarial_accuracy = None
stage4_v3_heuristic_adversarial_accuracy = None

if STAGE4_V3_RECORDS:
    supported_gold = [r for r in STAGE4_V3_RECORDS
                       if r["claim_type"] == "human_gold" and r["human_gold_status"] == "Supported"]
    if supported_gold:
        n_correct = sum(1 for r in supported_gold if r["nli_status"] == "Supported")
        stage4_v3_supported_agreement = n_correct / len(supported_gold)

    insufficient_gold = [r for r in STAGE4_V3_RECORDS
                          if r["claim_type"] == "human_gold" and r["human_gold_status"] == "Insufficient Evidence"]
    if insufficient_gold:
        n_correct = sum(1 for r in insufficient_gold if r["nli_status"] == "Insufficient Evidence")
        stage4_v3_insufficient_agreement = n_correct / len(insufficient_gold)

    adversarial_rows = [r for r in STAGE4_V3_RECORDS if r["claim_type"] == "adversarial"]
    if adversarial_rows:
        n_nli_correct = sum(1 for r in adversarial_rows if r["nli_status"] == "Unsupported")
        stage4_v3_adversarial_accuracy = n_nli_correct / len(adversarial_rows)
        n_heur_correct = sum(1 for r in adversarial_rows if r["heuristic_status"] == "Unsupported")
        stage4_v3_heuristic_adversarial_accuracy = n_heur_correct / len(adversarial_rows)

stage4_v3_improves_over_heuristic = None
if stage4_v3_adversarial_accuracy is not None and stage4_v3_heuristic_adversarial_accuracy is not None:
    adversarial_delta = stage4_v3_adversarial_accuracy - stage4_v3_heuristic_adversarial_accuracy
    if adversarial_delta > 0:
        stage4_v3_improves_over_heuristic = True
        adversarial_delta_text = (f"IMPROVEMENT: NLI adversarial accuracy is "
                                   f"{adversarial_delta:+.1%} higher than the heuristic on this set.")
    elif adversarial_delta < 0:
        stage4_v3_improves_over_heuristic = False
        adversarial_delta_text = (f"DECLINE: NLI adversarial accuracy is "
                                   f"{adversarial_delta:+.1%} lower than the heuristic on this set.")
    else:
        stage4_v3_improves_over_heuristic = False
        adversarial_delta_text = "NO CHANGE: NLI and heuristic adversarial accuracy are equal on this set."
else:
    adversarial_delta = None
    adversarial_delta_text = "Not computable (no adversarial claims evaluated)."

print("=" * 60)
print("STAGE 15 / STAGE 4 V3 METRICS")
print("=" * 60)
sa = stage4_v3_supported_agreement
ia = stage4_v3_insufficient_agreement
ha = stage4_v3_heuristic_adversarial_accuracy
na = stage4_v3_adversarial_accuracy
print(f"Supported agreement (n_gold=24):                {'n/a' if sa is None else f'{sa:.1%}'}")
print(f"Insufficient Evidence agreement (n_gold=5):      {'n/a' if ia is None else f'{ia:.1%}'}")
print(f"Heuristic adversarial accuracy (n_gold=7):       {'n/a' if ha is None else f'{ha:.1%}'}")
print(f"NLI adversarial accuracy (n_gold=7):             {'n/a' if na is None else f'{na:.1%}'}")
print(f"Heuristic -> NLI change on adversarial set:      {adversarial_delta_text}")
print("-" * 60)
print("No Unsupported metric is reported from the 29-claim human reference set: it has")
print("zero gold Unsupported examples. Unsupported is scored only via the 7 adversarial claims.")
print("=" * 60)


In [ ]:
# --- Stage 15g: Export (new file names -- Stage 4 V2 files untouched) -------
from collections import Counter

stage4_v3_df = pd.DataFrame(STAGE4_V3_RECORDS)

# STAGE4V3_NLI_CSV / STAGE4V3_NLI_JSON / STAGE4V3_NLI_SUMMARY now come from Stage 1.

stage4_v3_df.to_csv(STAGE4V3_NLI_CSV, index=False)
with open(STAGE4V3_NLI_JSON, "w") as f:
    json.dump(STAGE4_V3_RECORDS, f, indent=2)

for existing in (STAGE4V2_REFERENCE_CSV, STAGE4V2_REFERENCE_JSON,
                  STAGE4V2_ADVERSARIAL_CSV, STAGE4V2_ADVERSARIAL_JSON):
    assert STAGE4V3_NLI_CSV != existing and STAGE4V3_NLI_JSON != existing, (
        "Stage 15 output must not overwrite any Stage 13/14 output file."
    )

total_evaluated = len(STAGE4_V3_RECORDS)
human_gold_evaluated = sum(1 for r in STAGE4_V3_RECORDS if r["claim_type"] == "human_gold")
adversarial_evaluated = sum(1 for r in STAGE4_V3_RECORDS if r["claim_type"] == "adversarial")
nli_status_counts = Counter(r["nli_status"] for r in STAGE4_V3_RECORDS)

stage15_status = "PASS" if (stage4_v3_prereqs_ok and total_evaluated == 36) else "FAIL"

STAGE4_V3_LIMITATIONS = [
    "This NLI model is general-language and not clinically validated.",
    "NLI checks textual consistency with the retrieved evidence chunk, not complete "
    "clinical truth or guideline compliance beyond what that chunk states.",
    "The human gold/reference set is single-annotator (not independently double-checked).",
    "Both the heuristic and NLI verifiers were evaluated on only 7 adversarial claims -- "
    "too small a sample to generalise an accuracy estimate beyond this test set.",
]

stage4_v3_summary = {
    "stage": "Stage 15 / Stage 4 V3 -- NLI-based claim verification",
    "model_name": STAGE4_V3_MODEL_NAME,
    "decision_thresholds": {"entailment_supported": NLI_ENTAILMENT_THRESHOLD, "contradiction_unsupported": NLI_CONTRADICTION_THRESHOLD},
    "total_claims_evaluated": total_evaluated,
    "human_gold_claims_evaluated": human_gold_evaluated,
    "adversarial_claims_evaluated": adversarial_evaluated,
    "nli_status_counts": dict(nli_status_counts),
    "supported_agreement": stage4_v3_supported_agreement,
    "insufficient_evidence_agreement": stage4_v3_insufficient_agreement,
    "heuristic_adversarial_accuracy": stage4_v3_heuristic_adversarial_accuracy,
    "nli_adversarial_accuracy": stage4_v3_adversarial_accuracy,
    "adversarial_accuracy_delta_nli_minus_heuristic": adversarial_delta,
    "nli_improves_over_heuristic": stage4_v3_improves_over_heuristic,
    "stage15_status": stage15_status,
    "limitations": STAGE4_V3_LIMITATIONS,
}
with open(STAGE4V3_NLI_SUMMARY, "w") as f:
    json.dump(stage4_v3_summary, f, indent=2)

print(f"Saved {len(stage4_v3_df)} rows to {STAGE4V3_NLI_CSV} and {STAGE4V3_NLI_JSON}")
print(f"Saved summary to {STAGE4V3_NLI_SUMMARY}")
print("(Stage 4 V2 files stage4v2_reference_claim_comparison.csv/.json are untouched.)")

print("\n" + "=" * 60)
print("STAGE 15 -- FINAL STATUS")
print("=" * 60)
print(f"Stage 15 status:                   {stage15_status}")
if stage4_v3_improves_over_heuristic is None:
    print("Does NLI improve over heuristic?   not computable")
else:
    verdict = "YES" if stage4_v3_improves_over_heuristic else "NO"
    ha2 = stage4_v3_heuristic_adversarial_accuracy
    na2 = stage4_v3_adversarial_accuracy
    print(f"Does NLI improve over heuristic?   {verdict} (heuristic={ha2:.1%}, NLI={na2:.1%})")
print("-" * 60)
print("Limitations:")
for lim in STAGE4_V3_LIMITATIONS:
    print(f"  - {lim}")
print("=" * 60)

stage4_v3_df


---
# Stage 16 — Sensitivity Analysis + Heuristic-OR-NLI Ensemble (recovered from archive)

**Input:** `STAGE4_V3_RECORDS` (Stage 15, already-computed NLI scores).
**Output:** `SENSITIVITY_ROWS`, `sensitivity_threshold_analysis.csv`,
`sensitivity_summary.json`, `sensitivity_report.md`.

Re-applies the Stage 15 NLI decision rule to the **already-computed** entailment /
contradiction / neutral scores at three thresholds (0.70 primary, 0.50 and 0.45
exploratory) and builds an exploratory heuristic-OR-NLI ensemble rule. **No new model
inference is run in this stage** -- everything here is arithmetic over Stage 15's
stored scores. The 0.50 / 0.45 thresholds and the ensemble rule were chosen after
seeing the primary 0.70 result, so they are reported as exploratory, post-hoc
figures, not a second validated result.


In [ ]:
# --- Stage 16a: Re-derive the decision rule + reproducibility check ---------
def classify_from_stored_scores(entailment: float, contradiction: float, neutral: float, threshold: float) -> str:
    """
    Re-applies the Stage 15 (Stage 4 V3) decision rule to already-computed
    NLI scores at an arbitrary threshold. No new model inference.
    """
    scores = {"entailment": entailment, "contradiction": contradiction, "neutral": neutral}
    strongest = max(scores, key=scores.get)
    if entailment >= threshold and strongest == "entailment":
        return "Supported"
    elif contradiction >= threshold and strongest == "contradiction":
        return "Unsupported"
    else:
        return "Insufficient Evidence"


# Correctness check: re-deriving at threshold 0.70 must exactly reproduce
# the nli_status already stored in STAGE4_V3_RECORDS.
_reproduction_mismatches = [
    r for r in STAGE4_V3_RECORDS
    if classify_from_stored_scores(r["entailment_score"], r["contradiction_score"],
                                    r["neutral_score"], 0.70) != r["nli_status"]
]

print("=" * 60)
print("STAGE 16 -- REPRODUCIBILITY CHECK (threshold 0.70)")
print("=" * 60)
print(f"Rows checked: {len(STAGE4_V3_RECORDS)}")
print(f"Mismatches against the already-stored primary nli_status: {len(_reproduction_mismatches)}")
sensitivity_reproduction_ok = len(_reproduction_mismatches) == 0
print(f"Reproduction check: {'PASS' if sensitivity_reproduction_ok else 'FAIL'}")
print("=" * 60)


In [ ]:
# --- Stage 16b: Threshold sweep + heuristic-OR-NLI ensemble -----------------
# SENSITIVITY_THRESHOLDS now comes from Stage 1 (still [0.70, 0.50, 0.45]).

SENSITIVITY_ROWS = []
for r in STAGE4_V3_RECORDS:
    row = {
        "question": r["question"],
        "atomic_claim": r["atomic_claim"],
        "claim_type": r["claim_type"],
        "human_gold_status": r["human_gold_status"],
        "heuristic_status": r["heuristic_status"],
        "entailment_score": r["entailment_score"],
        "contradiction_score": r["contradiction_score"],
        "neutral_score": r["neutral_score"],
    }
    for t in SENSITIVITY_THRESHOLDS:
        row[f"nli_status_thr_{t:.2f}"] = classify_from_stored_scores(
            r["entailment_score"], r["contradiction_score"], r["neutral_score"], t
        )
    # Exploratory heuristic-OR-NLI ensemble, using the PRIMARY (0.70) NLI status.
    row["ensemble_status"] = (
        "Unsupported"
        if (r["heuristic_status"] == "Unsupported" or row["nli_status_thr_0.70"] == "Unsupported")
        else row["nli_status_thr_0.70"]
    )
    SENSITIVITY_ROWS.append(row)


def _adversarial_accuracy(rows, status_field):
    adv = [r for r in rows if r["claim_type"] == "adversarial"]
    correct = sum(1 for r in adv if r[status_field] == "Unsupported")
    return correct, len(adv)


sensitivity_results = {}
for t in SENSITIVITY_THRESHOLDS:
    correct, total = _adversarial_accuracy(SENSITIVITY_ROWS, f"nli_status_thr_{t:.2f}")
    sensitivity_results[f"threshold_{t:.2f}"] = {
        "adversarial_correct": correct, "adversarial_total": total,
        "adversarial_accuracy": correct / total if total else None,
    }

ensemble_correct, ensemble_total = _adversarial_accuracy(SENSITIVITY_ROWS, "ensemble_status")
sensitivity_results["ensemble"] = {
    "adversarial_correct": ensemble_correct, "adversarial_total": ensemble_total,
    "adversarial_accuracy": ensemble_correct / ensemble_total if ensemble_total else None,
}

print("=" * 60)
print("STAGE 16 -- ADVERSARIAL ACCURACY BY THRESHOLD (reproduced from stored scores)")
print("=" * 60)
for key, res in sensitivity_results.items():
    acc = res["adversarial_accuracy"]
    acc_txt = "" if acc is None else f" = {acc*100:.2f}%"
    print(f"  {key:16}: {res['adversarial_correct']}/{res['adversarial_total']}{acc_txt}")
print("=" * 60)


In [ ]:
# --- Stage 16c: Export (new file names -- Stage 15 files untouched) ---------
sensitivity_df = pd.DataFrame(SENSITIVITY_ROWS)
# SENSITIVITY_CSV now comes from Stage 1.
sensitivity_df.to_csv(SENSITIVITY_CSV, index=False)

for existing in (STAGE4V3_NLI_CSV, STAGE4V3_NLI_JSON, STAGE4V3_NLI_SUMMARY):
    assert SENSITIVITY_CSV != existing, "Stage 16 output must not overwrite any Stage 15 output file."

sensitivity_summary = {
    "stage": "Stage 16 -- alternative NLI decision thresholds and heuristic-OR-NLI ensemble",
    "method": "Re-applied the Stage 15 (Stage 4 V3) decision rule to already-computed entailment/"
              "contradiction/neutral scores per claim -- no new NLI model inference was run.",
    "reproduction_check_against_primary_0_70": "PASS" if sensitivity_reproduction_ok else "FAIL",
    "primary_reference_threshold": NLI_ENTAILMENT_THRESHOLD,
    "threshold_0.70": sensitivity_results["threshold_0.70"],
    "threshold_0.50": sensitivity_results["threshold_0.50"],
    "threshold_0.45": sensitivity_results["threshold_0.45"],
    "ensemble": sensitivity_results["ensemble"],
    "status": "exploratory / post-hoc -- thresholds 0.50 and 0.45 and the ensemble rule were "
              "selected after seeing the primary (0.70) result; they are not calibrated on a "
              "separate development set and are not the primary validated result.",
}
# SENSITIVITY_SUMMARY now comes from Stage 1.
with open(SENSITIVITY_SUMMARY, "w") as f:
    json.dump(sensitivity_summary, f, indent=2)

t70 = sensitivity_results["threshold_0.70"]
t50 = sensitivity_results["threshold_0.50"]
t45 = sensitivity_results["threshold_0.45"]
ens = sensitivity_results["ensemble"]

sensitivity_report_md = f"""# Sensitivity Analysis -- Stage 15 (Stage 4 V3) NLI Decision Threshold

Reproduced in this notebook run, directly from the already-computed NLI scores
(no new model inference).

Reproduction check against the primary threshold (0.70): **{"PASS" if sensitivity_reproduction_ok else "FAIL"}**

| Variant | Adversarial accuracy |
|---|---|
| Threshold 0.70 (primary) | {t70['adversarial_correct']}/{t70['adversarial_total']} = {t70['adversarial_accuracy']*100:.2f}% |
| Threshold 0.50 (exploratory) | {t50['adversarial_correct']}/{t50['adversarial_total']} = {t50['adversarial_accuracy']*100:.2f}% |
| Threshold 0.45 (exploratory) | {t45['adversarial_correct']}/{t45['adversarial_total']} = {t45['adversarial_accuracy']*100:.2f}% |
| Heuristic OR NLI ensemble (exploratory review rule) | {ens['adversarial_correct']}/{ens['adversarial_total']} = {ens['adversarial_accuracy']*100:.2f}% |

**These exploratory figures are post-hoc, not calibrated on a separate
development set, and are not presented as the primary validated result.**
The primary result remains NLI at threshold 0.70 (see Stage 15).
"""
# SENSITIVITY_REPORT_MD now comes from Stage 1.
with open(SENSITIVITY_REPORT_MD, "w") as f:
    f.write(sensitivity_report_md)

stage16_status = "PASS" if sensitivity_reproduction_ok and len(SENSITIVITY_ROWS) == len(STAGE4_V3_RECORDS) else "FAIL"

print(f"Saved {SENSITIVITY_CSV} ({len(sensitivity_df)} rows)")
print(f"Saved {SENSITIVITY_SUMMARY}")
print(f"Saved {SENSITIVITY_REPORT_MD}")
print(f"\nStage 16 status: {stage16_status}")
print("\nThis is the final stage. Pipeline now covers 16 stages end-to-end.")
